# Setup

In [1]:
import os
os.environ['PATH'] = '/venv/main/bin:' + os.environ['PATH']

In [3]:
!pip install -r requirements.txt
# !pip install torch triton pytest 
# !pip install liger_kernel  --no-build-isolation

  Using cached iniconfig-2.3.0-py3-none-any.whl.metadata (2.5 kB)
  Using cached pluggy-1.6.0-py3-none-any.whl.metadata (4.8 kB)
Using cached pluggy-1.6.0-py3-none-any.whl (20 kB)
Using cached iniconfig-2.3.0-py3-none-any.whl (7.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [liger-kernel] [liger-kernel]


In [4]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.backends.cuda.flash_sdp_enabled())

import sys
print(sys.version)

2.11.0+cu130
13.0
True
3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:50:00) [GCC 14.3.0]


In [6]:
#!pip install nvidia-cuda-runtime-cu12

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 5.5 MB/s  0:00:01m0:00:01:00:01m


In [7]:
# pre-build flash-attn
# !pip install "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.11/flash_attn-2.8.3+cu12torch2.11cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.7/253.7 MB 7.6 MB/s  0:00:34m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [flash-attn]2 [flash-attn]


In [8]:
# %%bash
# mkdir -p /root/.jupyter/lab/user-settings/@jupyterlab/notebook-extension/
# cat > /root/.jupyter/lab/user-settings/@jupyterlab/notebook-extension/tracker.jupyterlab-settings << 'EOF'
# {
#     "codeCellConfig": {
#         "codeFolding": true,
#         "foldGutter": true,
#         "lineNumbers": true
#     }
# }
# EOF

# `/efficient_model`

In [35]:
# !mkdir efficient_model

mkdir: efficient_model: File exists


In [37]:
# %%writefile efficient_model/__init__.py

# from efficient_model.norm import RMSNorm, RMSNormFunction, rmsnorm_forward, rmsnorm_backward
# from efficient_model.swiglu import (
#     SwiGLUFeedForward, MemoryEfficientSwiGLUMLP,
#     swiglu_forward, swiglu_backward
# )
# from efficient_model.attention import RotaryPositionalEmbedding, MultiHeadAttention
# from efficient_model.loss import CrossEntropyLoss
# from efficient_model.transformer import EfficientTransformer, TransformerBlock

# __all__ = [
#     "RMSNorm", "RMSNormFunction", "rmsnorm_forward", "rmsnorm_backward",
#     "SwiGLUFeedForward", "MemoryEfficientSwiGLUMLP",
#     "swiglu_forward", "swiglu_backward",
#     "RotaryPositionalEmbedding", "MultiHeadAttention",
#     "cross_entropy_loss", "CrossEntropyLoss",
#     "EfficientTransformer", "TransformerBlock",
# ]


Overwriting efficient_model/__init__.py


In [38]:
# %%writefile efficient_model/attention.py
# """
# Multi-head attention with flash attention implementation and RoPE.
# """

# import math
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# from config import TransformerConfig
# torch.backends.cuda.enable_mem_efficient_sdp(True)

# from flash_attn.layers.rotary import apply_rotary_emb

# class RotaryPositionalEmbedding(nn.Module):
#     def __init__(self, head_dim: int, max_seq_len: int = 2048, theta: float = 10000.0):
#         super().__init__()
#         self.head_dim = head_dim
#         self.max_seq_len = max_seq_len
#         self.theta = theta
#         inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim)) # (S, head_dim//2)
#         self.register_buffer('inv_freq', inv_freq, persistent=False)
#         self._build_cache(max_seq_len)
    
#     def _build_cache(self, seq_len: int):
#         positions = torch.arange(seq_len, device=self.inv_freq.device)
#         freqs = torch.outer(positions, self.inv_freq)  # (S, head_dim//2)
#         self.register_buffer('cos', freqs.cos(), persistent=False)
#         self.register_buffer('sin', freqs.sin(), persistent=False)
    
#     def forward(self, q: torch.Tensor, k: torch.Tensor, seq_len: int) -> tuple[torch.Tensor, torch.Tensor]:
#         """
#         Apply rotary positional embedding to q and k.
        
#         Args:
#             q: (B, num_heads, S, head_dim)
#             k: (B, num_heads, S, head_dim)
#             seq_len: sequence length (must be <= max_seq_len)
            
#         Returns:
#             q_rotated, k_rotated with same shapes
#         """
#         cos = self.cos[:seq_len]  # (S, head_dim//2)
#         sin = self.sin[:seq_len]
#         q_rotated = apply_rotary_emb(q.transpose(1, 2), cos, sin).transpose(1, 2)
#         k_rotated = apply_rotary_emb(k.transpose(1, 2), cos, sin).transpose(1, 2)     
#         return q_rotated, k_rotated
    

# class MultiHeadAttention(nn.Module):
#     def __init__(self, config: TransformerConfig):
#         super().__init__()
#         self.config = config
#         self.hidden_dim = config.hidden_dim
#         self.num_heads = config.num_heads
#         self.head_dim = config.hidden_dim // config.num_heads
#         self.qkv_proj = nn.Linear(config.hidden_dim, 3 * config.hidden_dim, bias=False)
#         self.out_proj = nn.Linear(config.hidden_dim, config.hidden_dim, bias=False)
#         self.rope = RotaryPositionalEmbedding(
#             head_dim=self.head_dim,
#             max_seq_len=config.max_seq_len,
#             theta=config.rope_theta,
#         )
#         self.dropout = nn.Dropout(config.dropout)

#     def forward(
#         self, 
#         x: torch.Tensor, 
#         attention_mask: torch.Tensor | None = None,
#     ) -> torch.Tensor:
#         B, S, H = x.shape
#         q, k, v = torch.chunk(self.qkv_proj(x), 3, dim=-1)
#         q = q.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
#         k = k.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
#         v = v.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
#         q, k = self.rope(q, k, S)
#         v = v.contiguous() # detach it to free qkv_proj(x)
#         out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout.p if self.training else 0.0)
#         out = out.transpose(1, 2).contiguous().view(B, S, H)
#         out = self.out_proj(out)
#         return out

Overwriting efficient_model/attention.py


In [39]:
# %%writefile efficient_model/loss.py

# """
# Fused linear cross-entropy for causal LM.

# Uses Liger's chunked implementation: processes lm_head projection
# and CE loss together without materializing the full B×S×V logits tensor.
# """

# import torch
# import torch.nn as nn

# from liger_kernel.transformers import LigerFusedLinearCrossEntropyLoss

# class CrossEntropyLoss(nn.Module):
#     def __init__(self, ignore_index: int = -100):
#         super().__init__()
#         self.ignore_index = ignore_index
#         self.fused_lce = LigerFusedLinearCrossEntropyLoss(reduction='mean', ignore_index=ignore_index)

#     def forward(self, hidden_states, weight, labels) -> torch.Tensor:
#         hidden_states = hidden_states[:, :-1].contiguous()
#         labels = labels[:, 1:].contiguous()
#         B, S, H = hidden_states.shape
#         return self.fused_lce(weight, hidden_states.view(B * S, H), labels.view(B * S))

Overwriting efficient_model/loss.py


In [40]:
# %%writefile efficient_model/norm.py
# """
# Fused RMSNorm with torch.compile and custom autograd.

# Compiled forward runs square/mean/rsqrt/scale in a single fused kernel.
# Custom backward saves only (x, rsqrt, weight) instead of every intermediate,
# reducing saved-tensor memory from ~8·B·S·H (fp32) to B·S·H (model dtype).
# """

# import torch
# import torch.nn as nn


# @torch.compile
# def rmsnorm_forward(x, weight, eps):
#     input_dtype = x.dtype
#     x_fp32 = x.float()
#     mean_sq = (x_fp32 * x_fp32).mean(dim=-1, keepdim=True)
#     rsqrt = torch.rsqrt(mean_sq + eps)
#     normalized = x_fp32 * rsqrt
#     scale = 1.0 + weight.float()
#     output = (normalized * scale).to(input_dtype)
#     return output, rsqrt, x


# @torch.compile
# def rmsnorm_backward(grad_output, rsqrt, x, weight):
#     x_fp32 = x.float()
#     grad_fp32 = grad_output.float()
#     scale = 1.0 + weight.float()
#     x_norm = x_fp32 * rsqrt
#     scaled_grad = grad_fp32 * scale
#     inner = (scaled_grad * x_norm).mean(dim=-1, keepdim=True)
#     x_grad = rsqrt * (scaled_grad - x_norm * inner)
#     all_dims_but_last = tuple(range(grad_output.dim() - 1))
#     weight_grad = (scaled_grad * x_norm).sum(dim=all_dims_but_last, keepdim=False)
#     return x_grad.to(x.dtype), weight_grad


# class RMSNormFunction(torch.autograd.Function):
#     @staticmethod
#     def forward(ctx, x, weight, eps):
#         output, rsqrt, x_orig = rmsnorm_forward(x, weight, eps)
#         ctx.save_for_backward(rsqrt, x_orig, weight)
#         return output

#     @staticmethod
#     def backward(ctx, grad_output):
#         rsqrt, x_orig, weight = ctx.saved_tensors
#         x_grad, weight_grad = rmsnorm_backward(grad_output, rsqrt, x_orig, weight)
#         return x_grad, weight_grad, None


# class RMSNorm(nn.Module):
#     def __init__(self, hidden_dim: int, eps: float = 1e-6):
#         super().__init__()
#         self.eps = eps
#         self.weight = nn.Parameter(torch.zeros(hidden_dim))

#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         return RMSNormFunction.apply(x, self.weight, self.eps)

Overwriting efficient_model/norm.py


In [41]:
# %%writefile efficient_model/swiglu.py
# """
# gpt-oss style SwiGLU Feed-Forward Network with fusion on triton and optimized checkpointing

# Reference SwiGLU implementation:
# https://github.com/linkedin/Liger-Kernel/blob/main/src/liger_kernel/ops/swiglu.py
# """

# import torch
# import torch.nn as nn
# import triton
# import triton.language as tl

# from liger_kernel.ops.utils import calculate_settings, ensure_contiguous

# from torch.amp import custom_fwd, custom_bwd



# @triton.jit
# def silu(x, alpha):
#     return x * tl.sigmoid(x * alpha)


# @triton.jit
# def _swiglu_forward_kernel(
#     a_ptr, b_ptr, c_ptr, stride, 
#     alpha: float, limit: float,
#     n_cols: tl.constexpr, BLOCK_SIZE: tl.constexpr
# ):
#     # a = gate, b = up
#     program_id = tl.program_id(0).to(tl.int64)

#     a_ptr += program_id * stride
#     b_ptr += program_id * stride
#     c_ptr += program_id * stride

#     col_offsets = tl.arange(0, BLOCK_SIZE)
#     mask = col_offsets < n_cols

#     # sigmoid requires fp32
#     a_row = tl.load(a_ptr + col_offsets, mask=mask, other=0).to(tl.float32) 
#     b_row = tl.load(b_ptr + col_offsets, mask=mask, other=0)
        
#     a_row = tl.minimum(a_row, limit)
#     b_row = tl.clamp(b_row, min=-limit, max=limit)
    
#     c_row = silu(a_row, alpha).cast(b_row.dtype) * (b_row + 1)
#     tl.store(c_ptr + col_offsets, c_row, mask=mask)


# @triton.jit
# def _swiglu_backward_kernel(
#     dc_ptr, a_ptr, b_ptr, stride, 
#     alpha: float, limit: float, n_cols: 
#     tl.constexpr, BLOCK_SIZE: tl.constexpr
# ):
#     program_id = tl.program_id(0).to(tl.int64)

#     dc_ptr += program_id * stride
#     a_ptr += program_id * stride
#     b_ptr += program_id * stride

#     col_offsets = tl.arange(0, BLOCK_SIZE)
#     mask = col_offsets < n_cols

#     dc_row = tl.load(dc_ptr + col_offsets, mask=mask, other=0)

#     # sigmoid requires fp32
#     a_row = tl.load(a_ptr + col_offsets, mask=mask, other=0).to(tl.float32) 
#     b_row = tl.load(b_ptr + col_offsets, mask=mask, other=0)

#     a_row = tl.minimum(a_row, limit)
#     b_row = tl.clamp(b_row, min=-limit, max=limit)
    
#     # for c=[sigmoid(a*alpha)*a]*(b+1) we have dL/da=dL/dc*[alpha*dsigmoid/da+sigmoid]*(b+1)
#     sig_a = tl.sigmoid(a_row * alpha)
#     silu_a = (a_row * sig_a)
#     db_row = dc_row * silu_a
#     da_row = (b_row + 1) * dc_row * (alpha * silu_a * (1 - sig_a) + sig_a)

#     # clamp derivatives
#     da_row = tl.where(a_row > limit, 0.0, da_row)
#     db_row = tl.where((b_row < -limit) | (b_row > limit), 0.0, db_row)

#     # downcast gate, up only in the end to preserve precision
#     tl.store(a_ptr + col_offsets, da_row.cast(b_row.dtype), mask=mask)
#     tl.store(b_ptr + col_offsets, db_row.cast(b_row.dtype), mask=mask)
    
#     # compute swiglu in bf16 to match forward and grad of activation_out with activation_out itself
#     tl.store(dc_ptr + col_offsets, (silu_a.cast(b_row.dtype) * (b_row + 1)), mask=mask)


# def swiglu_forward(a, b, alpha, limit):
#     ori_shape = a.shape
#     n_cols = ori_shape[-1]
    
#     # we work with 1D vectors
#     a = a.view(-1, n_cols)
#     b = b.view(-1, n_cols)
#     c = torch.empty_like(a)
#     n_rows = a.shape[0]
    
#     BLOCK_SIZE, num_warps = calculate_settings(n_cols)
    
#     _swiglu_forward_kernel[(n_rows,)](
#         a,
#         b,
#         c,
#         c.stride(-2),
#         alpha=alpha,
#         limit=limit,
#         n_cols=n_cols,
#         BLOCK_SIZE=BLOCK_SIZE,
#         num_warps=num_warps,
#     )
#     return a, b, c.view(*ori_shape)


# def swiglu_backward(a, b, dc, alpha, limit):
#     ori_shape = dc.shape
#     n_cols = ori_shape[-1]
#     dc = dc.view(-1, n_cols)
#     n_rows = dc.shape[0]

#     BLOCK_SIZE, num_warps = calculate_settings(n_cols)

#     _swiglu_backward_kernel[(n_rows,)](
#         dc,
#         a,
#         b,
#         dc.stride(-2),
#         alpha=alpha,
#         limit=limit,
#         n_cols=n_cols,
#         BLOCK_SIZE=BLOCK_SIZE,
#         num_warps=num_warps,
#     )
#     return a.view(*ori_shape), b.view(*ori_shape)


# class MemoryEfficientSwiGLUMLP(torch.autograd.Function):
#     @staticmethod
#     @custom_fwd(device_type='cuda')
#     def forward(ctx, x, w_gate, w_up, w_down, alpha, limit):
#         gate = x @ w_gate.T
#         up = x @ w_up.T
#         gate, up, activation_out = swiglu_forward(gate, up, alpha, limit)
#         out = activation_out @ w_down.T

#         ctx.save_for_backward(x, gate, up, w_gate, w_up, w_down)
#         ctx.alpha = alpha
#         ctx.limit = limit

#         return out
    
#     @staticmethod
#     @custom_bwd(device_type='cuda')
#     def backward(ctx, out_grad):
#         """
#         Memory-efficient backward: reuse storage from forward (gate, up, x)
#         and overwrite activation_out_grad with activation_out via the Triton kernel.

#         Here we use gradient identities for Y = X @ W.T (G = dL/dY):
#         dL/dX = G @ W, dL/dW = G.T @ X.
#         """
#         x, gate, up, w_gate, w_up, w_down = ctx.saved_tensors
        
#         # first, for swiglu_backward we need the following gradient:
#         activation_out_grad = out_grad @ w_down
        
#         # second, we run backward for swiglu. Note, it overwrites every arguments!
#         gate_grad, up_grad = swiglu_backward(gate, up, activation_out_grad, ctx.alpha, ctx.limit)
        
#         # third, since backward just overwritten activation_out_grad with activation_out,
#         # we can calculate all remaining gradients just as linear layers:
#         # a) out = activation_out @ w_down.T
#         # b) gate = x @ w_gate.T
#         # c) up = x @ w_up.T
#         activation_out = activation_out_grad 
#         w_down_grad = out_grad.reshape(-1, out_grad.shape[-1]).T @ activation_out.reshape(-1, activation_out.shape[-1])        
#         w_gate_grad = gate_grad.reshape(-1, gate_grad.shape[-1]).T @ x.reshape(-1, x.shape[-1])
#         w_up_grad = up_grad.reshape(-1, up_grad.shape[-1]).T @ x.reshape(-1, x.shape[-1])

#         # reuse x storage for x_grad!
#         x_grad = x
#         x_grad.copy_(gate_grad @ w_gate)
#         x_grad += up_grad @ w_up
#         return x_grad, w_gate_grad, w_up_grad, w_down_grad, None, None


# class SwiGLUFeedForward(nn.Module):
#     def __init__(self, hidden_dim: int, intermediate_dim: int):
#         super().__init__()
#         self.hidden_dim = hidden_dim
#         self.intermediate_dim = intermediate_dim
#         self.alpha = 1.702
#         self.limit = 7.0
#         self.gate_proj = nn.Linear(hidden_dim, intermediate_dim, bias=False)
#         self.up_proj = nn.Linear(hidden_dim, intermediate_dim, bias=False)
#         self.down_proj = nn.Linear(intermediate_dim, hidden_dim, bias=False)

#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         return MemoryEfficientSwiGLUMLP.apply(
#             x, self.gate_proj.weight, self.up_proj.weight, self.down_proj.weight, self.alpha, self.limit
#         )


Overwriting efficient_model/swiglu.py


In [42]:
# %%writefile efficient_model/transformer.py
# """
# Transformer Model with all optimizations.
# """

# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# from config import TransformerConfig
# from efficient_model.norm import RMSNorm
# from efficient_model.swiglu import SwiGLUFeedForward
# from efficient_model.attention import MultiHeadAttention
# from efficient_model.loss import CrossEntropyLoss


# class TransformerBlock(nn.Module):
#     def __init__(self, config: TransformerConfig):
#         super().__init__()
#         self.ln1 = RMSNorm(config.hidden_dim, eps=config.rms_norm_eps)
#         self.attn = MultiHeadAttention(config)
#         self.ln2 = RMSNorm(config.hidden_dim, eps=config.rms_norm_eps)
#         self.ffn = SwiGLUFeedForward(config.hidden_dim, config.intermediate_dim)

#     def forward(
#         self,
#         x: torch.Tensor,
#         attention_mask: torch.Tensor | None = None,
#     ) -> torch.Tensor:
#         x = x + self.attn(self.ln1(x), attention_mask)
#         x = x + self.ffn(self.ln2(x))
#         return x


# class EfficientTransformer(nn.Module):
#     def __init__(self, config: TransformerConfig):
#         super().__init__()
#         self.config = config

#         self.embedding = nn.Embedding(config.vocab_size, config.hidden_dim)

#         self.layers = nn.ModuleList([
#             TransformerBlock(config) for _ in range(config.num_layers)
#         ])

#         self.ln_f = RMSNorm(config.hidden_dim, eps=config.rms_norm_eps)
#         self.lm_head = nn.Linear(config.hidden_dim, config.vocab_size, bias=False)
#         self.loss_fn = CrossEntropyLoss(ignore_index=-100)
#         self.apply(self._init_weights)

#     def _init_weights(self, module):
#         if isinstance(module, nn.Linear):
#             torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
#             if module.bias is not None:
#                 torch.nn.init.zeros_(module.bias)
#         elif isinstance(module, nn.Embedding):
#             torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

#     def forward(
#         self,
#         input_ids: torch.Tensor,
#         attention_mask: torch.Tensor | None = None,
#         labels: torch.Tensor | None = None,
#     ) -> torch.Tensor:
#         """Forward pass. Returns loss (scalar) if labels given, logits (B, S, V) otherwise."""
#         B, S = input_ids.shape
#         x = self.embedding(input_ids)

#         for layer in self.layers:
#             x = layer(x, attention_mask)

#         x = self.ln_f(x)
        
#         if labels is not None:
#             return self.loss_fn(x, self.lm_head.weight, labels)
#         else:
#             logits = self.lm_head(x)
#             return logits.float()


Overwriting efficient_model/transformer.py


`efficient_optimizer`

In [8]:
!mkdir efficient_optimizer

In [9]:
# %%writefile efficient_optimizer/ademamix.py

# """
# AdEMAMix optimizer with optimizations - fused kernels, foreach_* routines, etc.
# No master copy inside.
# """

# import math
 
# import torch
# from torch import Tensor
# from torch.distributed.tensor import DTensor
# from torch.optim import Optimizer
# from torch._higher_order_ops.foreach_map import foreach_map

 
# def linear_warmup_scheduler(step, alpha_end, alpha_start=0, warmup=1):
#     if step < warmup:
#         a = step / float(warmup)
#         return (1.0-a) * alpha_start + a * alpha_end
#     return alpha_end


# def linear_hl_warmup_scheduler(step, beta_end, beta_start=0, warmup=1):
#     def f(beta, eps=1e-8):
#         return math.log(0.5)/math.log(beta+eps)-1
#     def f_inv(t):
#         return math.pow(0.5, 1/(t+1))
#     if step < warmup:
#         a = step / float(warmup)
#         return f_inv((1.0-a) * f(beta_start) + a * f(beta_end))
#     return beta_end


# def _ademamix_single(
#     param, exp_avg_fast, exp_avg_slow, exp_avg_sq,
#     grad,
#     beta1, beta2, beta3, one_minus_beta3, alpha,
#     bias_correction1, bias_correction2,
#     lr, eps, weight_decay,
# ):
#     new_exp_avg_fast = exp_avg_fast * beta1 + grad * (1 - beta1)
#     new_exp_avg_sq = exp_avg_sq * beta2 + grad * grad * (1 - beta2)
#     new_exp_avg_slow = exp_avg_slow * beta3 + grad * one_minus_beta3

#     denom = (new_exp_avg_sq / bias_correction2).sqrt() + eps
#     update = (new_exp_avg_fast / bias_correction1 + alpha * new_exp_avg_slow) / denom
#     update = update + param * weight_decay
#     new_param = param + update * (-lr)

#     return new_param, new_exp_avg_fast, new_exp_avg_slow, new_exp_avg_sq


# @torch.compile(fullgraph=True)
# def ademamix_foreach_map_fn(
#     params, grads, exp_avg_fasts, exp_avg_slows, exp_avg_sqs,
#     beta1: float, beta2: float,
#     beta3: Tensor, one_minus_beta3: Tensor, alpha: Tensor,
#     bias_correction1: Tensor, bias_correction2: Tensor,
#     lr: float, eps: float, lmbda: float,
# ):
#     result = foreach_map(
#         _ademamix_single,
#         params, exp_avg_fasts, exp_avg_slows, exp_avg_sqs,
#         grads,
#         beta1, beta2, beta3, one_minus_beta3, alpha,
#         bias_correction1, bias_correction2,
#         lr, eps, lmbda,
#     )
#     new_params, new_fasts, new_slows, new_sqs = zip(*result)
#     torch._foreach_copy_(params, list(new_params))
#     torch._foreach_copy_(exp_avg_fasts, list(new_fasts))
#     torch._foreach_copy_(exp_avg_slows, list(new_slows))
#     torch._foreach_copy_(exp_avg_sqs, list(new_sqs))
 
# @torch.compile(fullgraph=True)
# def ademamix_foreach_fn(
#     params: list[Tensor],
#     grads: list[Tensor],
#     exp_avg_fasts: list[Tensor],
#     exp_avg_slows: list[Tensor],
#     exp_avg_sqs: list[Tensor],
#     beta1: float,
#     beta2: float,
#     beta3: Tensor,
#     one_minus_beta3: Tensor,
#     alpha: Tensor,
#     bias_correction1: Tensor,
#     bias_correction2: Tensor,
#     lr: Tensor,
#     eps: float,
#     lmbda: float,
# ):      
#     """
#     Workaround: _foreach_add_(tensors, other, alpha=scalar) requires
#     alpha to be a Python float, but we have to keep them as tensor
#     to avoid constant recompilation because of schedule. So for tensor-valued
#     alpha/beta3/lr we pre-multiply with _foreach_mul and add the result instead.
#     """
#     torch._foreach_mul_(exp_avg_fasts, beta1)
#     torch._foreach_mul_(exp_avg_sqs, beta2)
#     torch._foreach_mul_(exp_avg_slows, beta3)

#     torch._foreach_add_(exp_avg_fasts, grads, alpha=1-beta1)
#     torch._foreach_addcmul_(exp_avg_sqs, grads, grads, value=1-beta2)
    
#     torch._foreach_add_(exp_avg_slows, torch._foreach_mul(grads, one_minus_beta3))
    
#     m = torch._foreach_div(exp_avg_fasts, bias_correction1)
#     v = torch._foreach_div(exp_avg_sqs, bias_correction2)
#     torch._foreach_sqrt_(v)
#     torch._foreach_add_(v, eps)

#     torch._foreach_add_(m, torch._foreach_mul(exp_avg_slows, alpha))
    
#     torch._foreach_div_(m, v)
#     torch._foreach_add_(m, params, alpha=lmbda)

#     torch._foreach_add_(params, torch._foreach_mul(m, -lr))

 
# class AdEMAMix(Optimizer):
#     r"""Implements the AdEMAMix algorithm.

#     Arguments:
#         params (iterable): iterable of parameters to optimize or dicts defining
#             parameter groups
#         lr (float, optional): learning rate (default: 1e-3)
#         betas (Tuple[float, float, float], optional): coefficients used for computing
#             running averages of gradient and its square (default: (0.9, 0.999, 0.9999)) 
#             corresponding to beta_1, beta_2, beta_3 in AdEMAMix
#         alpha (float): AdEMAMix alpha coeficient mixing the slow and fast EMAs (default: 2)
#         beta3_warmup (int, optional): number of warmup steps used to increase beta3 (default: None)
#         alpha_warmup: (int, optional): number of warmup steps used to increase alpha (default: None)
#         eps (float, optional): term added to the denominator to improve
#             numerical stability (default: 1e-8)
#         weight_decay (float, optional): weight decay as in AdamW (default: 0)
#         use_foreach_map (bool, optional): use foreach_map (fused kernel) instead of foreach ops (default: False)
#     """

#     def __init__(self, params, lr=1e-3, betas=(0.9, 0.999, 0.9999), alpha=2.0, 
#                  beta3_warmup=None, alpha_warmup=None, eps=1e-8,
#                  weight_decay=0, use_foreach_map=False):
#         if not 0.0 <= lr:
#             raise ValueError("Invalid learning rate: {}".format(lr))
#         if not 0.0 <= eps:
#             raise ValueError("Invalid epsilon value: {}".format(eps))
#         if not 0.0 <= betas[0] < 1.0:
#             raise ValueError("Invalid beta parameter at index 0: {}".format(betas[0]))
#         if not 0.0 <= betas[1] < 1.0:
#             raise ValueError("Invalid beta parameter at index 1: {}".format(betas[1]))
#         if not 0.0 <= betas[2] < 1.0:
#             raise ValueError("Invalid beta parameter at index 2: {}".format(betas[2]))
#         if not 0.0 <= weight_decay:
#             raise ValueError("Invalid weight_decay value: {}".format(weight_decay))
#         if not 0.0 <= alpha:
#             raise ValueError("Invalid alpha value: {}".format(alpha))
#         defaults = dict(lr=lr, betas=betas, eps=eps, alpha=alpha, beta3_warmup=beta3_warmup,
#                         alpha_warmup=alpha_warmup, weight_decay=weight_decay)
#         super(AdEMAMix, self).__init__(params, defaults)
#         self.use_foreach_map = use_foreach_map

#     def __setstate__(self, state):
#         super(AdEMAMix, self).__setstate__(state)

#     @torch.no_grad()
#     def step(self, closure=None):
#         loss = None
#         if closure is not None:
#             with torch.enable_grad():
#                 loss = closure()
 
#         for group in self.param_groups:
#             lr = group["lr"]
#             lmbda = group["weight_decay"]
#             eps = group["eps"]
#             beta1, beta2, beta3_final = group["betas"]
#             beta3_warmup = group["beta3_warmup"]
#             alpha_final = group["alpha"]
#             alpha_warmup = group["alpha_warmup"]

#             params: list[Tensor] = []
#             grads: list[Tensor] = []
#             exp_avg_fasts: list[Tensor] = []
#             exp_avg_slows: list[Tensor] = []
#             exp_avg_sqs: list[Tensor] = []

#             for p in group["params"]:
#                 if p.grad is None:
#                     continue
#                 state = self.state[p]
#                 if len(state) == 0:
#                     state['exp_avg_slow'] = torch.zeros_like(p)
#                     state['exp_avg_fast'] = torch.zeros_like(p)
#                     state['exp_avg_sq'] = torch.zeros_like(p)

#                 params.append(p)
#                 grads.append(p.grad)
#                 exp_avg_fasts.append(state['exp_avg_fast'])
#                 exp_avg_slows.append(state['exp_avg_slow'])
#                 exp_avg_sqs.append(state['exp_avg_sq'])

#             if not params:
#                 continue

#             group['step'] = group.get('step', 0) + 1

#             bias_correction1 = 1 - beta1 ** group['step']
#             bias_correction2 = 1 - beta2 ** group['step']
            
#             if alpha_warmup is not None:
#                 alpha = linear_warmup_scheduler(group["step"], alpha_end=alpha_final, alpha_start=0, warmup=alpha_warmup)
#             else:
#                 alpha = alpha_final
            
#             if beta3_warmup is not None:
#                 beta3 = linear_hl_warmup_scheduler(group["step"], beta_end=beta3_final, beta_start=beta1, warmup=beta3_warmup)
#             else:
#                 beta3 = beta3_final

#             device = params[0].device
#             bias_correction1 = torch.tensor(bias_correction1, device=device)
#             bias_correction2 = torch.tensor(bias_correction2, device=device)
#             one_minus_beta3 = torch.tensor(1 - beta3, device=device)
#             alpha = torch.tensor(alpha, device=device)
#             beta3 = torch.tensor(beta3, device=device)
#             lr = torch.tensor(lr, device=device)

#             step_fn = ademamix_foreach_map_fn if self.use_foreach_map else ademamix_foreach_fn
#             step_fn(
#                 params=params, grads=grads, 
#                 exp_avg_fasts=exp_avg_fasts, exp_avg_slows=exp_avg_slows, exp_avg_sqs=exp_avg_sqs,
#                 beta1=beta1, beta2=beta2, beta3=beta3,
#                 one_minus_beta3=one_minus_beta3, alpha=alpha,
#                 bias_correction1=bias_correction1, bias_correction2=bias_correction2,
#                 lr=lr, eps=eps, lmbda=lmbda,
#             )
            
#         return loss

Writing efficient_optimizer/ademamix.py


# `/calculators`

In [10]:
# !mkdir calculators

mkdir: calculators: File exists


In [17]:
# %%writefile calculators/__init__.py



Overwriting calculators/__init__.py


In [18]:
# %%writefile calculators/base.py
# """
# Base classes and configurations for theoretical calculators.
# """

# from dataclasses import dataclass
# from abc import ABC, abstractmethod


# @dataclass
# class GPUSpec:
#     """GPU hardware specifications."""
#     name: str
#     memory_bandwidth_gbps: float  # GB/s
#     flops_bf16: float             # TFLOP/s for BF16
#     interconnect_bandwidth_gbps: float  # GB/s per direction (per GPU)


# H100_SXM = GPUSpec(
#     name="H100 SXM",
#     memory_bandwidth_gbps=2800,    # was 3350 -- using lower for sustained
#     flops_bf16=800,                # was 989 peak, using sustained
#     interconnect_bandwidth_gbps=400,  # NVLink bidirectional per GPU ~900, per direction ~450, conservative 400
# )

# T4 = GPUSpec(
#     name="T4",
#     memory_bandwidth_gbps=320,
#     flops_bf16=8.1,  # bf16 is emulated on fp32?
#     interconnect_bandwidth_gbps=32,  # PCIe Gen3 x16
# )

# RTX_3090 = GPUSpec(
#     name="RTX 3090",
#     memory_bandwidth_gbps=936,
#     flops_bf16=142,
#     interconnect_bandwidth_gbps=32,  # PCIe Gen4 x16
# )

# @dataclass
# class ModelConfig:
#     """Model configuration for calculations."""
#     vocab_size: int
#     hidden_dim: int
#     num_heads: int
#     num_layers: int
#     intermediate_dim: int
#     max_seq_len: int


# @dataclass
# class TrainingConfig:
#     """Training configuration."""
#     batch_size: int
#     seq_len: int
#     num_gpus: int
#     dtype_bytes: int = 2  # BF16 = 2 bytes


# class BaseCalculator(ABC):
#     """
#     Base class for theoretical calculators.
    
#     Conventions:
#     - "params" counted in number of scalar parameters
#     - memory in bytes
#     - time in milliseconds
#     - flops = multiply-accumulate counted as 2 ops (standard convention)
#     """

#     def __init__(
#         self,
#         model_config: ModelConfig,
#         training_config: TrainingConfig,
#         gpu_spec: GPUSpec,
#     ):
#         self.model = model_config
#         self.training = training_config
#         self.gpu = gpu_spec
#         # Shortcuts
#         self.B = self.training.batch_size
#         self.S = self.training.seq_len
#         self.H = self.model.hidden_dim
#         self.I = self.model.intermediate_dim
#         self.N = self.model.num_layers
#         self.V = self.model.vocab_size
#         self.G = self.training.num_gpus
#         self.k = (self.G - 1) / self.G  # fraction for ring all-reduce
#         self.h = self.model.num_heads
#         self.d = self.H // self.h

#     # TODO: add real utilizations coefficients
#     def roofline_time_ms(self, flops: int, memory_bytes: int) -> float:
#         """
#         Roofline model: time = max(compute_time, memory_time).
        
#         Why max not sum: GPU pipelines memory fetches with compute.
#         While ALUs process batch N, memory controller fetches batch N+1.
#         The slower side dominates; the faster hides behind it.
#         """
#         compute_time_s = flops / (self.gpu.flops_bf16 * 1e12)
#         memory_time_s = memory_bytes / (self.gpu.memory_bandwidth_gbps * 1e9)
#         return max(compute_time_s, memory_time_s) * 1000

#     def calculate_total_params(self) -> int:
#         """
#         Total parameters for a standard LLaMA-style transformer.
        
#         Per layer:
#           - Attention: Q, K, V, O projections = 4 * H * H
#           - MLP (SwiGLU): gate_proj + up_proj + down_proj = 3 * H * I
#           - 2x RMSNorm: 2 * H  (scale only, no bias)
        
#         Non-layer:
#           - Token embedding: V * H
#           - LM head: V * H
#           - Final RMSNorm: H
#         """
#         H, I, N, V = self.H, self.I, self.N, self.V

#         attention_params = 4 * H * H
#         mlp_params = 3 * H * I
#         ln_params = H
#         per_layer = attention_params + mlp_params + 2 * ln_params

#         embed_params = V * H
#         lm_head_params = V * H
#         final_ln = H

#         return embed_params + lm_head_params + N * per_layer + final_ln

#     def _total_param_memory_bytes(self, param_dtype_bytes: int = 2) -> int:
#         """Total parameter memory in bytes (un-sharded)."""
#         return self.calculate_total_params() * param_dtype_bytes

#     def _total_gradient_memory_bytes(self, grad_dtype_bytes: int = 2) -> int:
#         """Total gradient memory in bytes (un-sharded)."""
#         return self.calculate_total_params() * grad_dtype_bytes

#     def _total_optimizer_memory_bytes(self, num_states: int = 2, state_dtype_bytes: int = 4) -> int:
#         """
#         Total optimizer state memory (un-sharded).
#         Subclass decides number of states, their type and master copy.
#         """
#         return self.calculate_total_params() * num_states * state_dtype_bytes

#     @abstractmethod
#     def calculate_param_memory(self) -> int:
#         """Parameter memory per GPU (bytes). Subclass decides sharding."""
#         pass

#     @abstractmethod
#     def calculate_gradient_memory(self) -> int:
#         """Gradient memory per GPU (bytes). Subclass decides sharding."""
#         pass

#     @abstractmethod
#     def calculate_optimizer_memory(self) -> int:
#         """Optimizer state memory per GPU (bytes). Subclass decides sharding."""
#         pass

#     @abstractmethod
#     def mlp_saved_tensors_bytes(self):
#         pass

#     @abstractmethod
#     def attention_saved_tensors_bytes(self):
#         pass

#     @abstractmethod
#     def norm_saved_tensors_bytes(self):
#         pass

#     @abstractmethod
#     def non_layer_saved_tensors_bytes(self):
#         pass
    
#     def calculate_activation_memory(self):
#         B, S, H = self.B, self.S, self.H
#         layer = (
#             self.mlp_saved_tensors_bytes() + 
#             self.attention_saved_tensors_bytes() + 
#             2 * self.norm_saved_tensors_bytes()
#             # residuals aren saved as inputs
#         )

#         return self.N * layer + self.non_layer_saved_tensors_bytes()

#     @abstractmethod
#     def _peak_transient_bytes(self):
#         """Subclass decides materialisation"""
#         pass
        
#     def calculate_peak_memory(self) -> int:
#         """Peak memory during fwd+bwd."""
#         params = self.calculate_param_memory()
#         saved = self.calculate_activation_memory()
#         optim = self.calculate_optimizer_memory()
#         grads = self.calculate_gradient_memory()
#         transient = self._peak_transient_bytes()
#         return params + saved + optim + grads + transient

#     def _embedding_breakdown(self):
#         """
#         Read: B*S int64 indices + V*H embedding table
#         Write: B*S*H output
#         FLOPs: negligible
#         """
#         B, S, H, V = self.B, self.S, self.H, self.V
#         dtype = self.training.dtype_bytes
#         int64_dtype = 8
#         mem_bytes = (B * S * int64_dtype) + (V * H * dtype) + (B * S * H * dtype)
#         return 0, mem_bytes
    
#     def time_embedding_ms(self) -> float:
#         """Embedding lookup: purely memory-bound (gather operation)."""
#         flops, mem_bytes = self._embedding_breakdown()
#         return self.roofline_time_ms(flops=flops, memory_bytes=mem_bytes)

#     @abstractmethod
#     def _attention_breakdown(self):
#         """Subclass decides flash attention"""
#         pass
        
#     def time_attention_ms(self) -> float:
#         flops, mem_bytes = self._attention_breakdown()
#         return self.roofline_time_ms(flops=flops, memory_bytes=mem_bytes)
        
#     @abstractmethod
#     def _rms_norm_breakdown(self):
#         """Subclass decides fusion."""
#         pass
        
#     def time_rms_norm_ms(self) -> float:
#         flops, mem_bytes = self._rms_norm_breakdown()
#         return self.roofline_time_ms(flops=flops, memory_bytes=mem_bytes)
        
#     @abstractmethod
#     def _mlp_breakdown(self):
#         """SwiGLU MLP timing in ms. Subclass decides fusion."""
#         pass
        
#     def time_mlp_ms(self) -> float:
#         flops, mem_bytes = self._mlp_breakdown()
#         return self.roofline_time_ms(flops=flops, memory_bytes=mem_bytes)

#     @abstractmethod
#     def _lm_head_breakdown(self):
#         """Subclass decides materialization."""
#         pass
    
#     def time_lm_head_ms(self) -> float:
#         flops, mem_bytes = self._lm_head_breakdown()
#         return self.roofline_time_ms(flops=flops, memory_bytes=mem_bytes)

#     @abstractmethod
#     def _loss_breakdown(self):
#         """Subclass decides materialization."""
#         pass
    
#     def time_loss_ms(self) -> float:    
#         flops, mem_bytes = self._loss_breakdown()
#         return self.roofline_time_ms(flops=flops, memory_bytes=mem_bytes)

#     def time_forward_pass_ms(self) -> float:
#         """Single GPU time, no dependency on communication."""
#         total = self.time_embedding_ms()
#         for _ in range(self.N):
#             total += self.time_rms_norm_ms()
#             total += self.time_attention_ms()
#             total += self.time_rms_norm_ms()
#             total += self.time_mlp_ms()
#         total += self.time_rms_norm_ms()
#         total += self.time_lm_head_ms()
#         total += self.time_loss_ms()
#         return total

#     def time_backward_pass_ms(self) -> float:
#         return 2.0 * self.time_forward_pass_ms()

#     def time_forward_backward_ms(self) -> float:
#         return self.time_forward_pass_ms() + self.time_backward_pass_ms()

#     # ── Communication ─────────────────────────────────────────────────

#     @abstractmethod
#     def calculate_communication_volume(self) -> int:
#         """Total communication volume per step (bytes). Subclass decides communication."""
#         pass

#     @abstractmethod
#     def time_communication_ms(self) -> float:
#         """Communication time per step (ms). Subclass decides communication."""
#         pass

#     @abstractmethod
#     def overlap_efficiency(self) -> float:
#         """Fraction of communication overlapped with compute (0.0-1.0). Subclass decides communication."""
#         pass

#     @abstractmethod
#     def time_total_step_ms(self) -> float:
#         """Total step time = compute + (1 - overlap) × comm. Subclass decides communication."""
#         pass

Overwriting calculators/base.py


In [19]:
# %%writefile calculators/baseline_calculator.py
# """
# Baseline Calculator: DDP (DistributedDataParallel).

# DDP model:
# - Full model replica on each GPU (params, grads, optimizer states)
# - Communication: all-reduce gradients during backward
# - No parameter sharding
# - All activations saved (no recomputation)
# """

# from calculators.base import BaseCalculator


# class BaselineCalculator(BaseCalculator):
#     def calculate_param_memory(self) -> int:
#         """
#         DDP: full params on each GPU.
        
#         Mixed precision training: params stored in bf16 (2 bytes) for forward/backward,
#         master copy in fp32 (4 bytes) is counted in optimizer.
#         """
#         dtype = self.training.dtype_bytes
#         return self.calculate_total_params() * dtype

#     def calculate_gradient_memory(self) -> int:
#         """
#         DDP: full gradients on each GPU.
#         Gradients in bf16 (same dtype as forward pass).
#         """
#         dtype = self.training.dtype_bytes
#         return self.calculate_total_params() * dtype

#     def calculate_optimizer_memory(self) -> int:
#         """
#         AdEMAMix: 3 states (m, v, nu) in model type, no master weights, no sharding.
#         """
#         total_params = self.calculate_total_params()
#         dtype = self.training.dtype_bytes
#         return 3 * total_params * dtype

#     def mlp_saved_tensors_bytes(self):
#         """          
#         MLP (SwiGLU) saved tensors (in baseline each op saves its inputs)
#         1. gate_proj(x): saves x                          -> B*S*H
#         2. up_proj(x): saves link to x                    -> 0
#         3. gate.clamp(): saves gate                       -> B*S*I
#         4. up.clamp(): saves up                           -> B*S*I
#         5. gate * alpha: saves link to gate               -> 0 gate is already saved
#         6. sigmoid(gate*alpha): saves sigmoid output      -> B*S*I
#         7. gate * sigmoid: saves link to gate, sigmoid    -> B*S*I
#         8. (up+1) * glu: saves (up+1), glu                -> 2*B*S*I
#         9. down_proj(intermediate): saves intermediate    -> B*S*I
#         """
#         B, S, H, I, h = self.B, self.S, self.H, self.I, self.h
#         return  (1 * B * S * H + 7 * B * S * I) * self.training.dtype_bytes

#     def attention_saved_tensors_bytes(self):
#         """
#         From BaselineAttention.forward():
#         1. q_proj(x), k_proj(x), v_proj(x): saves x once (same ptr)    -> B*S*H
#         2. rope(q, k): saves q, k and cos, sin                         -> 2*B*S*H + ~0
#         3. q * scale:   -> saved SCALAR                                -> ~0
#         4. matmul(q_scaled, k.T): saves q_scaled, (k is view)          -> 1*B*S*H
#         5. masked_fill_: in-place, negligible                          -> ~0
#         6. softmax: saves OUTPUT                                       -> B*h*S*S
#         7. dropout: saves mask (bool)                                  -> ~0
#         8. matmul(attn_weights, v): saves link to softmax out          -> 0
#         9. out_proj(out): saves input                                  -> B*S*H
#         """
#         B, S, H, h = self.B, self.S, self.H, self.h
#         dtype = self.training.dtype_bytes
#         return (5 * B * S * H * dtype + 1 * B * h * S * S * dtype)

        
#     def norm_saved_tensors_bytes(self):
#         """RMSNorm (×2 per layer): x, x_normalised (all in fp32)"""
#         B, S, H = self.B, self.S, self.H
#         return 2 * B * S * H * 4

#     def non_layer_saved_tensors_bytes(self):
#         """
#         Non-layer:
#           - Embedding output: B*S*H
#           - Final norm input + stats
#           - LM head input: B*S*H
#           - cross-entropy saves softmax OUTPUT (in fp32, since logits are casted)
#           """
#         B, S, H, V = self.B, self.S, self.H, self.V
#         embed_indices = B * S * 8
#         final_norm_input = B * S * H * self.training.dtype_bytes
#         lm_head_input = B * S * H * self.training.dtype_bytes
#         ce_softmax = B * S * V * 4
#         return embed_indices + final_norm_input + lm_head_input + ce_softmax

#     def _peak_transient_bytes(self):
#         """
#         Peak transient is:
#         - attention - scores, attn_weights, weight and input/output tmp grads
#         - ce - fp32 softmax probs, logits fp32 gradients
#         - logits - lm_head grads, input / output grads
#         Note: logits in baseline are saved outside, so they eat memory anyway!
#         """
#         B, S, H, V, h = self.B, self.S, self.H, self.V, self.h
#         dtype = self.training.dtype_bytes
#         # score + attn_weight + weight grad + input grad + output grad
#         attn_peak = 2 * B * h * S * S * dtype + 3 * H * H * dtype + 2 * B * H * S * dtype
#         # fp32 saved softmax from ce + fp32 input grads
#         ce_peak = 2 * B * S * V * 4
#         # lm_head grad, lm_head input grad, lm_head output fp32 grad
#         logits_peak = B * V * dtype + B * H * S * dtype + B * S * V * 4
#         # saved outside in forward
#         out_logits = B * S * V * 4
#         return out_logits + max(attn_peak, ce_peak, logits_peak)
        
#     # ── Timing ────────────────────────────────────────────────────────
#     # DDP: no per-layer communication in forward.
#     def _attention_breakdown(self):
#         """        
#         Operations:
#           1. Q = X @ Wq  (B,S,H) × (H,H) -> (B,S,H)       : 2*B*S*H*H flops
#           2. K = X @ Wk  : same
#           3. V = X @ Wv  : same
#           4. scores = Q @ K^T  (B,h,S,d) × (B,h,d,S) -> (B,h,S,S) : 2*B*h*S*S*d = 2*B*S*S*H
#           5. attn_weights = softmax(scores)  : ~5*B*h*S*S (max, shift, exp, sum_exp, division)
#           6. context = weights @ V  (B,h,S,S) × (B,h,S,d) -> (B,h,S,d) : 2*B*S*S*H
#           7. out = context @ Wo  (B,S,H) × (H,H) -> (B,S,H) : 2*B*S*H*H
        
#         Memory:
#           Read weights: 4*H*H (Wq,Wk,Wv,Wo)
#           Read/write activations at each step (TWICE FOR SOFTMAX)
#         """
#         B, S, H, h, d = self.B, self.S, self.H, self.h, self.d
#         dtype = self.training.dtype_bytes

#         # q, k, v, o + Q @ K.T + softmax + scores @ V
#         flops = 4 * 2 * B * S * H * H + 5 * B * h * S * S + 2 * 2 * B * S * S * H 

#         # Memory (read inputs and write output for each operation)
#         weight_read = 4 * H * H * dtype
#         qkv_mem = 3 * (B * S * H * dtype + H * H * dtype + B * S * H * dtype)
#         score_mem = 2 * B * S * H * dtype + B * h * S * S * dtype
#         softmax_mem = 3 * B * h * S * S * dtype # softmax reads twice!
#         context_mem = B * h * S * S * dtype + B * S * H * dtype + B * S * H * dtype
#         out_mem = B * S * H * dtype + H * H * dtype + B * S * H * dtype
#         total_mem = qkv_mem + score_mem + softmax_mem + context_mem + out_mem
#         return flops, total_mem

#     def _mlp_breakdown(self):
#         """
#         Operations:
#           1. gate = X @ W_gate  (B,S,H) × (H,I) -> (B,S,I)  : 2*B*S*H*I
#           2. up   = X @ W_up    (B,S,H) × (H,I) -> (B,S,I)  : 2*B*S*H*I
#           3. down = act @ W_down (B,S,I) × (I,H) -> (B,S,H) : 2*B*S*I*H
#           * act  = SiLU(gate) * up  (elementwise)          : ~3*B*S*I (sigmoid, mul, mul)
#         """
#         B, S, H, I = self.B, self.S, self.H, self.I
#         dtype = self.training.dtype_bytes
#         flops = 6 * B * S * I * H
#         mem = (2 * B * S * H + 2 * H * I + 2 * B * S * I + # gate & up
#                4 * B * S * I + # clamped gate, up
#                2 * B * S * I + # gate * alpha
#                2 * B * S * I + # sigmoid
#                3 * B * S * I + # x*sigmoid
#                2 * B * S * I + # up + 1               
#                3 * B * S * I + # result * (up + 1)
#                B * S * I + H * I + B * S * H) * dtype # down_proj   
#         return flops, mem

#     def _rms_norm_breakdown(self):
#         """No fusion. 
#         Flops: FLOPs: 4×B×S×H (square, sum, div, mul)
#         Memory: read input twice to square and scale (B*S*H), write output (B*S*H)
#         """
#         B, S, H = self.B, self.S, self.H
#         dtype = self.training.dtype_bytes
#         fp32 = 4
#         mem = (B*S*H * dtype +       # read input fp16
#                B*S*H * fp32 +        # write x_fp32
#                B*S*H * fp32 +        # read x_fp32 (square)
#                B*S*H * fp32 +        # write x_squared
#                B*S*H * fp32 +        # read x_squared (mean)
#                B*S*H * fp32 +        # read x_fp32 (normalize)
#                B*S*H * fp32 +        # write normalized
#                B*S*H * fp32 +        # read normalized
#                B*S*H * fp32 +        # write scaled fp32
#                B*S*H * dtype)        # write output fp16 (cast back)
#         flops = 4 * B * S * H
#         return flops, mem
        
#     def _lm_head_breakdown(self):
#         """
#         Flops: 2 * B * S * H * V
#         Memory: read input (B*S*H) + weight (H*V), write output (B*S*V)
#         """
#         B, S, H, V = self.B, self.S, self.H, self.V
#         dtype = self.training.dtype_bytes
#         flops = 2 * B * S * H * V
#         mem = (B * S * H + H * V + B * S * V) * dtype
#         return flops, mem
        
#     def _loss_breakdown(self):
#         """
#         Memory: ~read logits twice for softmax (B*S*V), write probs, read probs
#         FLOPs: ~5*B*S*V (max, shift, exp, sum, div)
#         """
#         B, S, V = self.B, self.S, self.V
#         dtype = self.training.dtype_bytes
#         mem = 4 * B * S * V * dtype
#         flops = 5 * B * S * V
#         return flops, mem

#     # ── Communication (DDP) ───────────────────────────────────────────

#     def calculate_communication_volume(self) -> int:
#         """
#         DDP all-reduce gradient volume.
        
#         Ring all-reduce sends each element twice (reduce-scatter + all-gather),
#         each transferring (G-1)/G of the data.
        
#         Total bytes through each GPU's link:
#             2 × (G-1)/G × gradient_size_bytes
#         """
#         dtype = self.training.dtype_bytes
#         grad_bytes = self.calculate_total_params() * dtype
#         return int(2 * self.k * grad_bytes)

#     def time_communication_ms(self) -> float:
#         """
#         DDP communication time (ms).
        
#         time = volume / bandwidth
#         """
#         volume = self.calculate_communication_volume()
#         time_s = volume / (self.gpu.interconnect_bandwidth_gbps * 1e9)
#         return time_s * 1000

#     def overlap_efficiency(self):
#         """overlaps only in backward"""
#         attn_f, attn_m = self._attention_breakdown()
#         mlp_f, mlp_m = self._mlp_breakdown()
#         norm_f, norm_m = self._rms_norm_breakdown()
#         emb_f, emb_m = self._embedding_breakdown()
#         lm_f, lm_m = self._lm_head_breakdown()
#         loss_f, loss_m = self._loss_breakdown()

#         # backward needs ~2 times more compute and memory
#         total_flops = 2 * (self.N * (attn_f + mlp_f + 2 * norm_f) +
#                            emb_f + norm_f + lm_f + loss_f)
#         total_mem = 2 * (self.N * (attn_m + mlp_m + 2 * norm_m) +
#                          emb_m + norm_m + lm_m + loss_m)
        
#         total_comm = self.calculate_communication_volume()
        
#         compute_ms = total_flops / (self.gpu.flops_bf16 * 1e12) * 1000
#         mem_ms = total_mem / (self.gpu.memory_bandwidth_gbps * 1e9) * 1000
#         comm_ms = total_comm / (self.gpu.interconnect_bandwidth_gbps * 1e9) * 1000
        
#         if compute_ms >= mem_ms + comm_ms:
#             return 1.0
#         else:
#             hidden = compute_ms - mem_ms
#             return max(0, hidden / comm_ms)
    
#     def time_total_step_ms(self):
#         fwd = self.time_forward_pass_ms()
#         bwd_compute = 2 * fwd
#         comm = self.time_communication_ms()
#         eff = self.overlap_efficiency()
#         exposed_comm = (1 - eff) * comm
#         return fwd + bwd_compute + exposed_comm

Overwriting calculators/baseline_calculator.py


In [20]:
# %%writefile calculators/efficient_calculator.py
# """
# Efficient Calculator: FSDP (Fully Sharded Data Parallelism).

# FSDP model:
# - Parameters, gradients, optimizer states ALL sharded across G GPUs
# - Forward: all-gather params for current layer, compute, discard unsharded copy
# - Backward: all-gather params again, compute grads, reduce-scatter grads
# - Communication buffers for all-gather and reduce-scatter (unsharded, bf16)
# - Selective activation recomputation (e.g., recompute attention instead of saving scores)

# Key differences from DDP baseline:
# - Memory: ~1/G for params+grads+optimizer, but need comm buffers
# - Communication: 3x volume (2× all-gather + 1× reduce-scatter) vs 2× (all-reduce)
# - Activations: reduced via selective recomputation (e.g., FlashAttention doesn't save S×S scores)
# """

# from calculators.base import BaseCalculator


# class EfficientCalculator(BaseCalculator):
#     def calculate_total_params(self) -> int:
#         """Same architecture as baseline."""
#         return super().calculate_total_params()

#     def calculate_param_memory(self) -> int:
#         """FSDP: params sharded across G GPUs in bf16. (I ignore buffer space!)"""
#         return self.calculate_total_params() * 2 // self.G

#     def calculate_gradient_memory(self) -> int:
#         """FSDP: after reduce-scatter, each GPU holds 1/G of gradients."""
#         return self.calculate_total_params() * 2 // self.G

#     def calculate_optimizer_memory(self) -> int:
#         """AdEMAMix: 3 states (m, v, nu) in model type, no master weights. Sharded."""
#         total_params = self.calculate_total_params()
#         dtype = self.training.dtype_bytes
#         return 3 * total_params * dtype // self.G

#     def mlp_saved_tensors_bytes(self):
#         """Fused SwiGLU: doesn't read/write intermediate tensors."""
#         B, S, H, I, h = self.B, self.S, self.H, self.I, self.h
#         return (B * S * H + 2 * B * S * I) * self.training.dtype_bytes # x, gate, up (swiglu is recalculated!)

#     def attention_saved_tensors_bytes(self):
#         """FlashAttention: saves only Q, K, V and the output (log sum exp is negligible)."""
#         B, S, H, I, h = self.B, self.S, self.H, self.I, self.h
#         return 5 * B * S * H * self.training.dtype_bytes # x, q, k, v, o
        
#     def norm_saved_tensors_bytes(self):
#         """Fused RMSNorm - stores only fp16 and recomputes"""
#         B, S, H = self.B, self.S, self.H
#         return B * S * H * self.training.dtype_bytes

#     def non_layer_saved_tensors_bytes(self):
#         """Fused cross-entropy: doesn't materialize logits."""
#         B, S, H = self.B, self.S, self.H
#         embed_indices = B * S * 8 # no need for embeddings!
#         final_norm_input = B * S * H * self.training.dtype_bytes
#         fused_ce_input = B * S * H * self.training.dtype_bytes  # hidden_states
#         fused_ce_labels = B * S * 8
#         return embed_indices + final_norm_input + fused_ce_input + fused_ce_labels

#     def _peak_transient_bytes(self):
#         """
#         Candidates:
#         - fused logits+ce
#         - attention
#         Note, in fsdp we also need to take into account need to unshard layers and prefetch next!
#         Moreover, because CE is fused, we shard embedding with lm_head, so they are both in memory.
#         """
#         V, B, S, H, I = self.V, self.B, self.S, self.H, self.I
#         dtype = self.training.dtype_bytes
        
#         # fused chunked CE
#         BT = B * (S - 1) 
#         inc_factor = -(-V // H)
#         chunk_size = 1 << (-(-BT // inc_factor) - 1).bit_length()
#         # chunked tmp tensor + input grad + lm_head grad
#         ce_peak = chunk_size * V * dtype + B * S * H * dtype + V * H * dtype
#         # ~4k * 16k * 2
#         # gradient for flash-attention out and input (Q, K, V)
#         flash_attn_peak = 4 * B * S * H * dtype
#         # 4 * 64 * 4k * 1k
#         # ~4k * 16k * 2
#         # ~2^27

#         # because ce is fused we need this
#         unsharded_root_units = 2 * H * V * dtype
        
#         if self.G > 1:
#             layer_params = 4 * H * H + 3 * H * I + 2 * H
#             embed_params = V * H
#             lm_head_params = V * H
#             largest_unit = max(layer_params, embed_params, lm_head_params)
#             # unsharded layer with gradients, prefetch
#             fsdp_buffers = (2 * largest_unit + layer_params) * dtype 
#         else:
#             fsdp_buffers = 0
        
#         return max(ce_peak, flash_attn_peak) + fsdp_buffers + unsharded_root_units

#     def _rms_norm_breakdown(self) -> float:
#         """
#         Fused RMSNorm: single kernel, one read + one write pass.

#         FLOPS: ~4 * (B*S*H) (sq, sum, div, mul)
#         Memory: ~read input (B*S*H) + write output (B*S*H)
#         The fusion saves ~2 extra passes over B*S*H that a naive implementation does.
#         """
#         B, S, H = self.B, self.S, self.H
#         dtype = self.training.dtype_bytes
#         flops = 4 * B * S * H  + 2 * B * S
#         return flops, 2 * B * S * H * dtype
    
#     def _attention_breakdown(self) -> float:
#         """
#         FlashAttention tiles the computation to stay in SRAM, avoiding
#         materializing the S×S attention matrix in HBM.
        
#         FLOPs: Same as standard attention (same math, different memory pattern):
#           QKV projections: 3 × 2*B*S*H*H
#           Score + context: 2 × 2*B*S*S*H  (still computed, just in tiles)
#           Output projection: 2*B*S*H*H
        
#         Memory (KEY DIFFERENCE):
#           - QKV projections: X is read once,
#           - Attention core: reads Q,K,V once, writes O once.
#             No S×S matrix written to HBM!
        
#         For the attention core specifically:
#           mem ~ 4 * B * S * H * dtype (Q,K,V read + O write)
#           flops ~ 2 × 2 * B * S * S * H (QK^T + scores@V, same as before)
#         """
#         B, S, H, I, h, d = self.B, self.S, self.H, self.I, self.h, self.d
#         dtype = self.training.dtype_bytes

#         flops = 4 * 2 * B * S * H * H + 2 * 2 * B * S * S * H # qkvo, Q@K.T, scores@V
#         mem = 4 * B * S * H * dtype 
#         return flops, mem

#     def _mlp_breakdown(self):
#         """
#         Fused SwiGLU MLP.
        
#         1. gate = x @ W_gate.T   read x(B×S×H) + W_gate(H×I), write gate(B×S×I)
#         2. up   = x @ W_up.T     read x(B×S×H) + W_up(H×I),   write up(B×S×I)
#         3. SwiGLU(gate, up)       read gate+up(2×B×S×I),        write act(B×S×I)  -- fused, no intermediates
#         4. out  = act @ W_down.T  read act(B×S×I) + W_down(I×H), write out(B×S×H)
        
#         FLOPs: 3 × 2×B×S×H×I  (3 matmuls, SwiGLU negligible)
#         Memory: 2×B×S×H + 2×H×I + 2×B×S×I  (gate+up)
#               + 2×B×S×I + B×S×I             (SwiGLU read+write)
#               + B×S×I + H×I + B×S×H         (down)
#         """
#         B, S, H, I = self.B, self.S, self.H, self.I
#         dtype = self.training.dtype_bytes
        
#         flops = 6 * B * S * H * I
#         mem = (2*B*S*H + 2*H*I + 2*B*S*I +   # gate & up
#                2*B*S*I + B*S*I +               # fused SwiGLU
#                B*S*I + I*H + B*S*H) * dtype    # down_proj
#         return flops, mem

#     def _lm_head_breakdown(self):
        
#         """
#         LM head with potential chunked computation for fused CE.
        
#         If using fused linear cross-entropy, the LM head projection
#         and loss are computed together in chunks over the vocab dimension.
#         The full B×S×V logits tensor is never materialized.
        
#         Flops: 2 * B * S * H * V  (same matmul)
#         Memory: read input (B*S*H) num_chunks (V/C) times, weights (H*V) once, write result
#         Negligible: ~ 2*B*S writes and reads for online softmax, NLL

#         Note: in forward it already calculates all for backward:
#         logits = input_chunk @ weight.T
#         grad_input = logits @ weight
#         grad_weight += logits.T @ input_chunk
#         But we won't calculate it here because we like bwd = 3 fwd heuristics
#         """
#         B, S, H, V = self.B, self.S, self.H, self.V
#         dtype = self.training.dtype_bytes
#         # note herea actually 3 matmuls happening, but it will break x3 bwd heuristics :(
#         flops = 2 * B * S * H * V 
#         BT = B * (S - 1) 
#         inc_factor = -(-V // H)
#         chunk_size = 1 << (-(-BT // inc_factor) - 1).bit_length()
#         num_chunks = B * S // chunk_size
#         mem = (2 * B * S * H              # read input twice + write grad_input
#             + 2 * num_chunks * H * V     # read weight twice per chunk
#             + H * V                      # write grad_weight
#             ) * dtype
#         return flops, mem
        
#     def _loss_breakdown(self) -> float:
#         """
#         Fused linear cross-entropy: loss computed inline with LM head chunks.
#         But if we keep its flops there, we overestimate backward path.
#         So let's just count NLL's flops here:
#          ~5 * B * S * V (max, exp, sum, sub, div)
#         """
#         B, S, V = self.B, self.S, self.V
#         flops = 5 * B * S * V 
#         return flops, 0

#     # ── Communication (FSDP) ──────────────────────────────────────────
#     def calculate_communication_volume(self) -> int:
#         """
#         Total FSDP communication:
#         = 2 × all-gather (forward + backward) + 1 × reduce-scatter (backward)
#         = 3 × (G-1)/G × total_params × dtype
#         """
#         total_params = self.calculate_total_params()
#         dtype = self.training.dtype_bytes
#         return 3 * int(self.k * total_params * dtype)

#     def time_communication_ms(self) -> float:
#         """FSDP total communication time (ms)."""
#         volume = self.calculate_communication_volume()
#         time_s = volume / (self.gpu.interconnect_bandwidth_gbps * 1e9)
#         return time_s * 1000

#     def overlap_efficiency(self) -> float:
#         """
#         FSDP overlap analysis.
        
#         FORWARD:
#           - All-gather of layer N+1 overlaps with compute of layer N
        
#         BACKWARD:
#           - All-gather of layer N-1 overlaps with backward compute of layer N
#           - Reduce-scatter of layer N overlaps with backward compute of layer N-1

#         Since bwd_time ~ 2x fwd_time and bwd_comm ~ 2x fwd_comm, 
#         they SHARE efficiency and we simply calculate logic for forward.
#         """
#         attn_f, attn_m = self._attention_breakdown()
#         mlp_f, mlp_m = self._mlp_breakdown()
#         norm_f, norm_m = self._rms_norm_breakdown()
#         emb_f, emb_m = self._embedding_breakdown()
#         lm_f, lm_m = self._lm_head_breakdown()
#         loss_f, loss_m = self._loss_breakdown()

#         total_flops = (self.N * (attn_f + mlp_f + 2 * norm_f) +
#                         emb_f + norm_f + lm_f + loss_f)
#         total_mem = (self.N * (attn_m + mlp_m + 2 * norm_m) +
#                         emb_m + norm_m + lm_m + loss_m)

#         # total communication is x3 
#         total_comm = self.calculate_communication_volume() / 3
        
#         compute_ms = total_flops / (self.gpu.flops_bf16 * 1e12) * 1000
#         mem_ms = total_mem / (self.gpu.memory_bandwidth_gbps * 1e9) * 1000
#         comm_ms = total_comm / (self.gpu.interconnect_bandwidth_gbps * 1e9) * 1000
        
#         if compute_ms >= mem_ms + comm_ms:
#             return 1.0
#         else:
#             hidden = compute_ms - mem_ms
#             return max(0, hidden / comm_ms)
    
#     def time_total_step_ms(self):
#         fwd = self.time_forward_pass_ms()
#         bwd_compute = 2 * fwd
#         comm = self.time_communication_ms()
#         eff = self.overlap_efficiency()
#         exposed_comm = (1 - eff) * comm
#         return fwd + bwd_compute + exposed_comm

Overwriting calculators/efficient_calculator.py


# `/` (train, use it last)

In [11]:
# %%writefile train.py
# """
# Training Script with DDP

# Usage:
#     # Single GPU
#     python train.py
    
#     # Multi-GPU with DDP
#     torchrun --nproc_per_node=2 train.py
# """

# import os
# import time
# import argparse
# from contextlib import nullcontext

# import torch
# import torch.distributed as dist
# from torch.nn.parallel import DistributedDataParallel as DDP
# from torch.utils.data import Dataset, DataLoader, DistributedSampler

# from config import TransformerConfig
# from model import BaselineTransformer
# from optimizer.ademamix import AdEMAMix


# class SyntheticDataset(Dataset):
#     """
#     Synthetic dataset generating random token sequences.
#     Used for benchmarking.
#     """
    
#     def __init__(
#         self, 
#         num_samples: int, 
#         seq_len: int, 
#         vocab_size: int,
#         seed: int = 42,
#     ):
#         self.num_samples = num_samples
#         self.seq_len = seq_len
#         self.vocab_size = vocab_size
#         self.seed = seed
    
#     def __len__(self):
#         return self.num_samples
    
#     def __getitem__(self, idx):
#         generator = torch.Generator().manual_seed(self.seed + idx)
#         tokens = torch.randint(
#             0, self.vocab_size, (self.seq_len,), generator=generator
#         )
#         return tokens


# def setup_distributed():
#     """Initialize distributed training if available."""
#     if 'RANK' in os.environ:
#         dist.init_process_group(backend='nccl')
#         rank = dist.get_rank()
#         world_size = dist.get_world_size()
#         local_rank = int(os.environ.get('LOCAL_RANK', 0))
#         torch.cuda.set_device(local_rank)
#         return rank, world_size, local_rank
#     else:
#         return 0, 1, 0


# def cleanup_distributed():
#     """Clean up distributed training."""
#     if dist.is_initialized():
#         dist.destroy_process_group()


# def get_lr(step: int, warmup_steps: int, max_lr: float, total_steps: int) -> float:
#     """Linear warmup followed by cosine decay."""
#     if step < warmup_steps:
#         return max_lr * step / warmup_steps

#     progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
#     return max_lr * 0.5 * (1.0 + torch.cos(torch.tensor(progress * 3.14159)).item())


# def train(args):
#     rank, world_size, local_rank = setup_distributed()
#     device = torch.device(f'cuda:{local_rank}' if torch.cuda.is_available() else 'cpu')
#     is_master = rank == 0

#     if is_master:
#         print(f"Training with {world_size} GPU(s)")
#         print(f"Device: {device}")

#     torch.manual_seed(args.seed)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed(args.seed)

#     config = TransformerConfig()
#     if is_master:
#         print(f"Model config: {config}")

#     model = BaselineTransformer(config).to(device)

#     if world_size > 1:
#         model = DDP(model, device_ids=[local_rank])

#     raw_model = model.module if world_size > 1 else model
#     num_params = sum(p.numel() for p in model.parameters())
#     if is_master:
#         print(f"Model parameters: {num_params:,}")

#     optimizer = AdEMAMix(
#         model.parameters(),
#         lr=args.learning_rate,
#         betas=(0.9, 0.999, 0.9999),
#         alpha=args.alpha,
#         beta3_warmup=args.beta3_warmup,
#         alpha_warmup=args.alpha_warmup,
#         weight_decay=args.weight_decay,
#     )

#     dataset = SyntheticDataset(
#         num_samples=args.num_samples,
#         seq_len=config.max_seq_len,
#         vocab_size=config.vocab_size,
#         seed=args.seed,
#     )

#     sampler = DistributedSampler(dataset, shuffle=True) if world_size > 1 else None
#     dataloader = DataLoader(
#         dataset,
#         batch_size=args.batch_size,
#         shuffle=(sampler is None),
#         sampler=sampler,
#         num_workers=args.num_workers,
#         pin_memory=True,
#     )

#     total_steps = args.num_epochs * len(dataloader)
#     warmup_steps = int(0.1 * total_steps)

#     if is_master:
#         print(f"\nTraining for {args.num_epochs} epochs ({total_steps} steps)")
#         print(f"Batch size: {args.batch_size} x {world_size} = {args.batch_size * world_size}")
#         print(f"Sequence length {config.max_seq_len}")
#         print("-" * 60)

#     scaler = torch.amp.GradScaler('cuda', enabled=args.use_amp)
#     autocast_ctx = torch.amp.autocast('cuda', dtype=torch.bfloat16) if args.use_amp else nullcontext()

#     model.train()
#     global_step = 0
#     total_tokens = 0
#     start_time = time.time()
#     log_interval_tokens = 0
#     log_interval_start = time.time()

#     for epoch in range(args.num_epochs):
#         if sampler is not None:
#             sampler.set_epoch(epoch)

#         epoch_loss = 0.0
#         epoch_steps = 0

#         for batch_idx, input_ids in enumerate(dataloader):
#             input_ids = input_ids.to(device)
#             labels = input_ids.clone()

#             lr = get_lr(global_step, warmup_steps, args.learning_rate, total_steps)
#             for param_group in optimizer.param_groups:
#                 param_group['lr'] = lr

#             optimizer.zero_grad()

#             with autocast_ctx:
#                 logits = raw_model(input_ids)
#                 loss = raw_model.compute_loss(logits, labels)

#             scaler.scale(loss).backward()

#             if args.grad_clip > 0:
#                 scaler.unscale_(optimizer)
#                 torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)

#             scaler.step(optimizer)
#             scaler.update()

#             batch_tokens = input_ids.numel() * world_size
#             total_tokens += batch_tokens
#             log_interval_tokens += batch_tokens
#             epoch_loss += loss.item()
#             epoch_steps += 1
#             global_step += 1

#             if is_master and global_step % args.log_interval == 0:
#                 elapsed = time.time() - log_interval_start
#                 tokens_per_sec = log_interval_tokens / elapsed
#                 avg_loss = epoch_loss / epoch_steps
                
#                 print(
#                     f"Step {global_step:5d} | "
#                     f"Epoch {epoch+1}/{args.num_epochs} | "
#                     f"Loss {avg_loss:.4f} | "
#                     f"LR {lr:.2e} | "
#                     f"Tokens/s {tokens_per_sec:,.0f}"
#                 )

#                 log_interval_tokens = 0
#                 log_interval_start = time.time()
        
#         if is_master:
#             avg_epoch_loss = epoch_loss / epoch_steps
#             print(f"Epoch {epoch+1} complete | Avg Loss: {avg_epoch_loss:.4f}")
    
#     total_time = time.time() - start_time

#     cleanup_distributed()
    
#     return model


# def main():
#     parser = argparse.ArgumentParser(description='Baseline Transformer Training')

#     parser.add_argument('--batch-size', type=int, default=4,
#                         help='Batch size per GPU')
#     parser.add_argument('--num-epochs', type=int, default=1,
#                         help='Number of training epochs')
#     parser.add_argument('--num-samples', type=int, default=1000,
#                         help='Number of synthetic samples')
#     parser.add_argument('--learning-rate', type=float, default=1e-4,
#                         help='Peak learning rate')
#     parser.add_argument('--weight-decay', type=float, default=0.1,
#                         help='Weight decay')
#     parser.add_argument('--alpha', type=float, default=2.0,
#                         help='AdEMAMix alpha coefficient for mixing slow and fast EMAs')
#     parser.add_argument('--beta3-warmup', type=int, default=None,
#                         help='Number of warmup steps for beta3')
#     parser.add_argument('--alpha-warmup', type=int, default=None,
#                         help='Number of warmup steps for alpha')
#     parser.add_argument('--grad-clip', type=float, default=1.0,
#                         help='Gradient clipping (0 to disable)')
#     parser.add_argument('--use-amp', action='store_true',
#                         help='Use automatic mixed precision')
#     parser.add_argument('--seed', type=int, default=42,
#                         help='Random seed')
#     parser.add_argument('--num-workers', type=int, default=4,
#                         help='DataLoader workers')
#     parser.add_argument('--log-interval', type=int, default=10,
#                         help='Log every N steps')
    
#     args = parser.parse_args()
#     train(args)


# if __name__ == '__main__':
#     main()

Writing train.py


In [12]:
# %%writefile efficient_train.py
# """
# Training Script with FSDP

# Usage:
#     # Single GPU
#     python train.py
    
#     # Multi-GPU with DDP
#     torchrun --nproc_per_node=2 train.py
# """

# import os
# import time
# import argparse
# from contextlib import nullcontext

# import torch
# import torch.distributed as dist
# from torch.utils.data import Dataset, DataLoader, DistributedSampler

# from config import TransformerConfig
# from efficient_model import EfficientTransformer, TransformerBlock
# from efficient_optimizer.ademamix import AdEMAMix


# from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
# from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy


# class SyntheticDataset(Dataset):
#     """
#     Synthetic dataset generating random token sequences.
#     Used for benchmarking.
#     """
    
#     def __init__(
#         self, 
#         num_samples: int, 
#         seq_len: int, 
#         vocab_size: int,
#         seed: int = 42,
#     ):
#         self.num_samples = num_samples
#         self.seq_len = seq_len
#         self.vocab_size = vocab_size
#         self.seed = seed
    
#     def __len__(self):
#         return self.num_samples
    
#     def __getitem__(self, idx):
#         generator = torch.Generator().manual_seed(self.seed + idx)
#         tokens = torch.randint(
#             0, self.vocab_size, (self.seq_len,), generator=generator
#         )
#         return tokens


# def setup_distributed():
#     """Initialize distributed training if available."""
#     if 'RANK' in os.environ:
#         dist.init_process_group(backend='nccl')
#         rank = dist.get_rank()
#         world_size = dist.get_world_size()
#         local_rank = int(os.environ.get('LOCAL_RANK', 0))
#         torch.cuda.set_device(local_rank)
#         return rank, world_size, local_rank
#     else:
#         return 0, 1, 0


# def cleanup_distributed():
#     """Clean up distributed training."""
#     if dist.is_initialized():
#         dist.destroy_process_group()


# def get_lr(step: int, warmup_steps: int, max_lr: float, total_steps: int) -> float:
#     """Linear warmup followed by cosine decay."""
#     if step < warmup_steps:
#         return max_lr * step / warmup_steps

#     progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
#     return max_lr * 0.5 * (1.0 + torch.cos(torch.tensor(progress * 3.14159)).item())


# def train(args):
#     rank, world_size, local_rank = setup_distributed()
#     device = torch.device(f'cuda:{local_rank}' if torch.cuda.is_available() else 'cpu')

    
#     is_master = rank == 0

#     if is_master:
#         print(f"Training with {world_size} GPU(s)")
#         print(f"Device: {device}")

#     torch.manual_seed(args.seed)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed(args.seed)

#     config = TransformerConfig()
#     if is_master:
#         print(f"Model config: {config}")

#     model = EfficientTransformer(config).to(device) # dtype?

#     if world_size > 1:
#         # todo:
#         # wrap each TransformerBlock as a separate FSDP unit so params are sharded per-layer, not as one flat blob
#         import functools
#         from torch.distributed.fsdp.wrap import transformer_auto_wrap_policy

#         policy = functools.partial(
#             transformer_auto_wrap_policy,
#             transformer_layer_cls={TransformerBlock}
#         )

#         mp = MixedPrecision(
#             param_dtype=torch.bfloat16,
#             reduce_dtype=torch.float32,
#             buffer_dtype=torch.bfloat16,
#         )
#         # DDP works with device_ids, FSDP only with device_id
#         model = FSDP(model, mixed_precision=mp, auto_wrap_policy=policy, device_id=local_rank)
#         autocast_ctx = nullcontext()
#     else:
#         autocast_ctx = torch.amp.autocast('cuda', dtype=torch.bfloat16)

#     raw_model = model.module if world_size > 1 else model
#     num_params = sum(p.numel() for p in model.parameters())
#     if is_master:
#         print(f"Model parameters: {num_params:,}")

#     optimizer = AdEMAMix(
#         model.parameters(),
#         lr=args.learning_rate,
#         betas=(0.9, 0.999, 0.9999),
#         alpha=args.alpha,
#         beta3_warmup=args.beta3_warmup,
#         alpha_warmup=args.alpha_warmup,
#         weight_decay=args.weight_decay,
#     )

#     dataset = SyntheticDataset(
#         num_samples=args.num_samples,
#         seq_len=config.max_seq_len,
#         vocab_size=config.vocab_size,
#         seed=args.seed,
#     )

#     sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True) if world_size > 1 else None
#     dataloader = DataLoader(
#         dataset,
#         batch_size=args.batch_size,
#         shuffle=(sampler is None),
#         sampler=sampler,
#         num_workers=args.num_workers,
#         pin_memory=True,
#     )

#     total_steps = args.num_epochs * len(dataloader)
#     warmup_steps = int(0.1 * total_steps)

#     if is_master:
#         print(f"\nTraining for {args.num_epochs} epochs ({total_steps} steps)")
#         print(f"Batch size: {args.batch_size} x {world_size} = {args.batch_size * world_size}")
#         print(f"Sequence length: {config.max_seq_len}")
#         print("-" * 60)


#     model.train()
#     global_step = 0
#     total_tokens = 0
#     start_time = time.time()
#     log_interval_tokens = 0
#     log_interval_start = time.time()

#     for epoch in range(args.num_epochs):
#         if sampler is not None:
#             sampler.set_epoch(epoch)

#         epoch_loss = 0.0
#         epoch_steps = 0

#         for batch_idx, input_ids in enumerate(dataloader):
#             input_ids = input_ids.to(device)
#             labels = input_ids.clone()

#             lr = get_lr(global_step, warmup_steps, args.learning_rate, total_steps)
#             for param_group in optimizer.param_groups:
#                 param_group['lr'] = lr

#             optimizer.zero_grad()
#             with autocast_ctx:
#                 loss = raw_model(input_ids, labels=labels)
#             loss.backward()

#             if args.grad_clip > 0:
#                 torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)

#             optimizer.step()

#             batch_tokens = input_ids.numel() * world_size
#             total_tokens += batch_tokens
#             log_interval_tokens += batch_tokens
#             epoch_loss += loss.item()
#             epoch_steps += 1
#             global_step += 1

#             if is_master and global_step % args.log_interval == 0:
#                 elapsed = time.time() - log_interval_start
#                 tokens_per_sec = log_interval_tokens / elapsed
#                 avg_loss = epoch_loss / epoch_steps
                
#                 print(
#                     f"Step {global_step:5d} | "
#                     f"Epoch {epoch+1}/{args.num_epochs} | "
#                     f"Loss {avg_loss:.4f} | "
#                     f"LR {lr:.2e} | "
#                     f"Tokens/s {tokens_per_sec:,.0f}"
#                 )

#                 log_interval_tokens = 0
#                 log_interval_start = time.time()
        
#         if is_master:
#             avg_epoch_loss = epoch_loss / epoch_steps
#             print(f"Epoch {epoch+1} complete | Avg Loss: {avg_epoch_loss:.4f}")
    
#     total_time = time.time() - start_time

#     cleanup_distributed()
    
#     return model


# def main():
#     parser = argparse.ArgumentParser(description='Baseline Transformer Training')

#     parser.add_argument('--batch-size', type=int, default=4,
#                         help='Batch size per GPU')
#     parser.add_argument('--num-epochs', type=int, default=1,
#                         help='Number of training epochs')
#     parser.add_argument('--num-samples', type=int, default=1000,
#                         help='Number of synthetic samples')
#     parser.add_argument('--learning-rate', type=float, default=1e-4,
#                         help='Peak learning rate')
#     parser.add_argument('--weight-decay', type=float, default=0.1,
#                         help='Weight decay')
#     parser.add_argument('--alpha', type=float, default=2.0,
#                         help='AdEMAMix alpha coefficient for mixing slow and fast EMAs')
#     parser.add_argument('--beta3-warmup', type=int, default=None,
#                         help='Number of warmup steps for beta3')
#     parser.add_argument('--alpha-warmup', type=int, default=None,
#                         help='Number of warmup steps for alpha')
#     parser.add_argument('--grad-clip', type=float, default=1.0,
#                         help='Gradient clipping (0 to disable)')
#     parser.add_argument('--use-amp', action='store_true',
#                         help='Use automatic mixed precision')
#     parser.add_argument('--seed', type=int, default=42,
#                         help='Random seed')
#     parser.add_argument('--num-workers', type=int, default=4,
#                         help='DataLoader workers')
#     parser.add_argument('--log-interval', type=int, default=10,
#                         help='Log every N steps')
    
#     args = parser.parse_args()
#     train(args)


# if __name__ == '__main__':
#     main()


Writing efficient_train.py


In [13]:
# %%writefile config.py
# from dataclasses import dataclass


# @dataclass
# class TransformerConfig:
#     vocab_size: int = 16000
#     hidden_dim: int = 512
#     num_heads: int = 8
#     num_layers: int = 6
#     intermediate_dim: int = 1024
#     max_seq_len: int = 4096
#     dropout: float = 0.0
#     rope_theta: float = 10000.0
#     rms_norm_eps: float = 1e-6
    
#     def __post_init__(self):
#         assert self.hidden_dim % self.num_heads == 0, \
#             f"hidden_dim ({self.hidden_dim}) must be divisible by num_heads ({self.num_heads})"


Writing config.py


# `/optimizer`

In [14]:
# !mkdir optimizer

In [15]:
# %%writefile optimizer/ademamix.py
# """
# Baseline implementation of AdEMAMix optimizer.
# Source: https://github.com/apple/ml-ademamix
# """
# import math
# import torch
# from torch.optim import Optimizer


# def linear_warmup_scheduler(step, alpha_end, alpha_start=0, warmup=1):
#     if step < warmup:
#         a = step / float(warmup)
#         return (1.0-a) * alpha_start + a * alpha_end
#     return alpha_end


# def linear_hl_warmup_scheduler(step, beta_end, beta_start=0, warmup=1):

#     def f(beta, eps=1e-8):
#         return math.log(0.5)/math.log(beta+eps)-1

#     def f_inv(t):
#         return math.pow(0.5, 1/(t+1))

#     if step < warmup:
#         a = step / float(warmup)
#         return f_inv((1.0-a) * f(beta_start) + a * f(beta_end))
#     return beta_end


# class AdEMAMix(Optimizer):
#     r"""Implements the AdEMAMix algorithm.

#     Arguments:
#         params (iterable): iterable of parameters to optimize or dicts defining
#             parameter groups
#         lr (float, optional): learning rate (default: 1e-3)
#         betas (Tuple[float, float, float], optional): coefficients used for computing
#             running averages of gradient and its square (default: (0.9, 0.999, 0.9999)) 
#             corresponding to beta_1, beta_2, beta_3 in AdEMAMix
#         alpha (float): AdEMAMix alpha coeficient mixing the slow and fast EMAs (default: 2)
#         beta3_warmup (int, optional): number of warmup steps used to increase beta3 (default: None)
#         alpha_warmup: (int, optional): number of warmup steps used to increase alpha (default: None)
#         eps (float, optional): term added to the denominator to improve
#             numerical stability (default: 1e-8)
#         weight_decay (float, optional): weight decay as in AdamW (default: 0)
#     """

#     def __init__(self, params, lr=1e-3, betas=(0.9, 0.999, 0.9999), alpha=2.0, 
#                  beta3_warmup=None, alpha_warmup=None,  eps=1e-8,
#                  weight_decay=0):
#         if not 0.0 <= lr:
#             raise ValueError("Invalid learning rate: {}".format(lr))
#         if not 0.0 <= eps:
#             raise ValueError("Invalid epsilon value: {}".format(eps))
#         if not 0.0 <= betas[0] < 1.0:
#             raise ValueError("Invalid beta parameter at index 0: {}".format(betas[0]))
#         if not 0.0 <= betas[1] < 1.0:
#             raise ValueError("Invalid beta parameter at index 1: {}".format(betas[1]))
#         if not 0.0 <= betas[2] < 1.0:
#             raise ValueError("Invalid beta parameter at index 2: {}".format(betas[2]))
#         if not 0.0 <= weight_decay:
#             raise ValueError("Invalid weight_decay value: {}".format(weight_decay))
#         if not 0.0 <= alpha:
#             raise ValueError("Invalid alpha value: {}".format(alpha))
#         defaults = dict(lr=lr, betas=betas, eps=eps, alpha=alpha, beta3_warmup=beta3_warmup,
#                         alpha_warmup=alpha_warmup, weight_decay=weight_decay)
#         super(AdEMAMix, self).__init__(params, defaults)

#     def __setstate__(self, state):
#         super(AdEMAMix, self).__setstate__(state)

#     @torch.no_grad()
#     def step(self, closure=None):
#         """Performs a single optimization step.

#         Arguments:
#             closure (callable, optional): A closure that reevaluates the model
#                 and returns the loss.
#         """
#         loss = None
#         if closure is not None:
#             with torch.enable_grad():
#                 loss = closure()

#         for group in self.param_groups:
            
#             lr = group["lr"]
#             lmbda = group["weight_decay"]
#             eps = group["eps"]
#             beta1, beta2, beta3_final = group["betas"]
#             beta3_warmup = group["beta3_warmup"]
#             alpha_final = group["alpha"]
#             alpha_warmup = group["alpha_warmup"]
        
#             for p in group['params']:
#                 if p.grad is None:
#                     continue
#                 grad = p.grad
#                 if grad.is_sparse:
#                     raise RuntimeError('AdEMAMix does not support sparse gradients.')

#                 state = self.state[p]

#                 # State initialization
#                 if len(state) == 0:
#                     state['step'] = 0
#                     # Exponential moving average of gradient values
#                     if beta1 != 0.0: # save memory in case beta1 is 0.0
#                         state['exp_avg_fast'] = torch.zeros_like(p, memory_format=torch.preserve_format)
#                     else: 
#                         state['exp_avg_fast'] = None
#                     state['exp_avg_slow'] = torch.zeros_like(p, memory_format=torch.preserve_format)
#                     # Exponential moving average of squared gradient values
#                     state['exp_avg_sq'] = torch.zeros_like(p, memory_format=torch.preserve_format)

#                 exp_avg_fast, exp_avg_slow, exp_avg_sq = state['exp_avg_fast'], state['exp_avg_slow'], state['exp_avg_sq']

#                 state['step'] += 1
#                 bias_correction1 = 1 - beta1 ** state['step']
#                 bias_correction2 = 1 - beta2 ** state['step']

#                 # Compute the effective alpha and beta3 in case warmup is used 
#                 if alpha_warmup is not None:
#                     alpha = linear_warmup_scheduler(state["step"], alpha_end=alpha_final, alpha_start=0, warmup=alpha_warmup)
#                 else:
#                     alpha = alpha_final
                
#                 if beta3_warmup is not None:
#                     beta3 = linear_hl_warmup_scheduler(state["step"], beta_end=beta3_final, beta_start=beta1, warmup=beta3_warmup)
#                 else:
#                     beta3 = beta3_final

#                 # Decay the first and second moment running average coefficient
#                 if beta1 != 0.0:
#                     exp_avg_fast.mul_(beta1).add_(grad, alpha=1 - beta1)
#                 else:
#                     exp_avg_fast = grad
#                 exp_avg_slow.mul_(beta3).add_(grad, alpha=1 - beta3)
#                 exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

#                 denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(eps)

#                 update = (exp_avg_fast.div(bias_correction1) + alpha * exp_avg_slow) / denom

#                 # decay
#                 update.add_(p, alpha=lmbda)

#                 p.add_(-lr * update)

#         return loss


Writing optimizer/ademamix.py


# `/model`

In [16]:
# !mkdir model

mkdir: model: File exists


In [21]:
# %%writefile model/__init__.py

# from model.norm import RMSNorm
# from model.swiglu import SwiGLUFeedForward
# from model.attention import RotaryPositionalEmbedding, MultiHeadAttention
# from model.loss import cross_entropy_loss
# from model.transformer import BaselineTransformer, TransformerBlock

# __all__ = [
#     "RMSNorm",
#     "SwiGLUFeedForward",
#     "RotaryPositionalEmbedding", "MultiHeadAttention",
#     "cross_entropy_loss",
#     "BaselineTransformer", "TransformerBlock",
# ]


Overwriting model/__init__.py


In [22]:
# %%writefile model/attention.py
# """
# Attention with RoPE
# """

# import math
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# from config import TransformerConfig


# class RotaryPositionalEmbedding(nn.Module):
#     """
#     Rotary Positional Embedding (RoPE).
#     """
    
#     def __init__(self, head_dim: int, max_seq_len: int = 2048, theta: float = 10000.0):
#         super().__init__()
#         self.head_dim = head_dim
#         self.max_seq_len = max_seq_len
#         self.theta = theta

#         inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
#         self.register_buffer('inv_freq', inv_freq, persistent=False)

#         self._build_cache(max_seq_len)
    
#     def _build_cache(self, seq_len: int):
#         """Build sin/cos cache up to seq_len."""
#         positions = torch.arange(seq_len, device=self.inv_freq.device)
#         freqs = torch.outer(positions, self.inv_freq)
#         emb = torch.cat([freqs, freqs], dim=-1)

#         self.register_buffer('cos', emb.cos().unsqueeze(0).unsqueeze(0), persistent=False)
#         self.register_buffer('sin', emb.sin().unsqueeze(0).unsqueeze(0), persistent=False)
    
#     def forward(self, q: torch.Tensor, k: torch.Tensor, seq_len: int) -> tuple[torch.Tensor, torch.Tensor]:
#         """
#         Apply rotary positional embedding to q and k.
        
#         Args:
#             q: (B, num_heads, S, head_dim)
#             k: (B, num_heads, S, head_dim)
#             seq_len: sequence length (must be <= max_seq_len)
            
#         Returns:
#             q_rotated, k_rotated with same shapes
#         """
#         assert seq_len <= self.max_seq_len, \
#             f"seq_len ({seq_len}) exceeds max_seq_len ({self.max_seq_len})"
        
#         cos = self.cos[:, :, :seq_len, :]
#         sin = self.sin[:, :, :seq_len, :]

#         q_rotated = self._apply_rotary(q, cos, sin)
#         k_rotated = self._apply_rotary(k, cos, sin)
        
#         return q_rotated, k_rotated
    
#     def _apply_rotary(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
#         """Apply rotary embedding to tensor x."""
#         orig_dtype = x.dtype
#         x1 = x[..., : x.shape[-1] // 2]
#         x2 = x[..., x.shape[-1] // 2 :]
#         rotated = torch.cat([-x2, x1], dim=-1)
#         return (x.float() * cos.float() + rotated.float() * sin.float()).to(orig_dtype)


# class MultiHeadAttention(nn.Module):
#     """
#     Multi-head attention with vanilla implementation and RoPE.
#     """
    
#     def __init__(self, config: TransformerConfig):
#         super().__init__()
#         self.config = config
#         self.hidden_dim = config.hidden_dim
#         self.num_heads = config.num_heads
#         self.head_dim = config.hidden_dim // config.num_heads

#         self.q_proj = nn.Linear(config.hidden_dim, config.hidden_dim, bias=False)
#         self.k_proj = nn.Linear(config.hidden_dim, config.hidden_dim, bias=False)
#         self.v_proj = nn.Linear(config.hidden_dim, config.hidden_dim, bias=False)
#         self.out_proj = nn.Linear(config.hidden_dim, config.hidden_dim, bias=False)

#         self.scale = 1.0 / math.sqrt(self.head_dim)
#         self.max_seq_len = self.config.max_seq_len
#         self.register_buffer(
#             'causal_mask',
#             torch.triu(torch.ones(self.config.max_seq_len, self.max_seq_len, dtype=torch.bool), diagonal=1),
#             persistent=False
#         )

#         self.rope = RotaryPositionalEmbedding(
#             head_dim=self.head_dim,
#             max_seq_len=config.max_seq_len,
#             theta=config.rope_theta,
#         )

#         self.dropout = nn.Dropout(config.dropout)

#     def forward(
#         self, 
#         x: torch.Tensor, 
#         attention_mask: torch.Tensor | None = None,
#     ) -> torch.Tensor:
#         B, S, H = x.shape

#         q = self.q_proj(x)
#         k = self.k_proj(x)
#         v = self.v_proj(x)

#         q = q.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
#         k = k.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
#         v = v.view(B, S, self.num_heads, self.head_dim).transpose(1, 2)

#         q, k = self.rope(q, k, S)

#         attn_weights = torch.matmul(q * self.scale, k.transpose(-2, -1)) # scale inside

#         if attention_mask is None:
#             causal_mask = self.causal_mask[:S,:S] 
#             attn_weights.masked_fill_(causal_mask, float('-inf')) # in-place
#         else:
#             attn_weights = attn_weights + attention_mask

#         attn_weights = F.softmax(attn_weights, dim=-1)
#         attn_weights = self.dropout(attn_weights)

#         out = torch.matmul(attn_weights, v)

#         out = out.transpose(1, 2).contiguous().view(B, S, H)
#         out = self.out_proj(out)

#         return out


Writing model/attention.py


In [23]:
# %%writefile model/loss.py
# """
# Cross Entropy Loss for Causal LM
# """

# import torch
# import torch.nn.functional as F


# def cross_entropy_loss(
#     logits: torch.Tensor,
#     labels: torch.Tensor,
#     ignore_index: int = -100,
# ) -> torch.Tensor:
#     """
#     Cross entropy loss for causal language modeling.
    
#     Shifts logits and labels for next-token prediction:
#     - logits[:, :-1] predicts labels[:, 1:]
    
#     Args:
#         logits: (B, S, vocab_size)
#         labels: (B, S)
#         ignore_index: label to ignore
#     """
#     shift_logits = logits[:, :-1, :].contiguous().float()
#     shift_labels = labels[:, 1:].contiguous()
    
#     return F.cross_entropy(
#         shift_logits.view(-1, shift_logits.size(-1)),
#         shift_labels.view(-1),
#         ignore_index=ignore_index,
#     )


Writing model/loss.py


In [24]:
# %%writefile model/norm.py
# """
# Zero-Centered RMSNorm
# """

# import torch
# import torch.nn as nn


# class RMSNorm(nn.Module):
#     """
#     Zero-Centered RMSNorm: y = x/rms(x) * (1 + weight), weight init to zeros.
#     """

#     def __init__(self, hidden_dim: int, eps: float = 1e-6):
#         super().__init__()
#         self.eps = eps
#         self.weight = nn.Parameter(torch.zeros(hidden_dim))

#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         x_fp32 = x.float()
#         x_squared = x_fp32 * x_fp32
#         mean_squared = x_squared.mean(dim=-1, keepdim=True)
#         mean_squared_eps = mean_squared + self.eps
#         rsqrt = torch.rsqrt(mean_squared_eps)
#         normalized = x_fp32 * rsqrt
#         output = normalized * (1.0 + self.weight.float())
#         return output.type_as(x)


Writing model/norm.py


In [25]:
# %%writefile model/swiglu.py
# """
# gpt-oss style SwiGLU Feed-Forward Network
# """

# import torch
# import torch.nn as nn


# class SwiGLUFeedForward(nn.Module):
#     """
#     gpt-oss style SwiGLU.
    
#     output = W_down @ ((up + 1) * gate * sigmoid(gate * alpha))
#     """
    
#     def __init__(self, hidden_dim: int, intermediate_dim: int):
#         super().__init__()
#         self.hidden_dim = hidden_dim
#         self.intermediate_dim = intermediate_dim
#         self.alpha = 1.702
#         self.limit = 7.0

#         self.gate_proj = nn.Linear(hidden_dim, intermediate_dim, bias=False)
#         self.up_proj = nn.Linear(hidden_dim, intermediate_dim, bias=False)
#         self.down_proj = nn.Linear(intermediate_dim, hidden_dim, bias=False)

#     def forward(self, x: torch.Tensor) -> torch.Tensor:
#         gate = self.gate_proj(x)
#         up = self.up_proj(x)
        
#         gate = gate.clamp(max=self.limit)
#         up = up.clamp(min=-self.limit, max=self.limit)
#         glu = gate * torch.sigmoid(gate * self.alpha)
#         intermediate = (up + 1) * glu
        
#         output = self.down_proj(intermediate)
#         return output


Writing model/swiglu.py


In [26]:
# %%writefile model/transformer.py
# """
# Baseline Transformer Model
# """

# import torch
# import torch.nn as nn

# from config import TransformerConfig
# from model.norm import RMSNorm
# from model.swiglu import SwiGLUFeedForward
# from model.attention import MultiHeadAttention
# from model.loss import cross_entropy_loss


# class TransformerBlock(nn.Module):
#     """Single transformer block."""
    
#     def __init__(self, config: TransformerConfig):
#         super().__init__()
#         self.ln1 = RMSNorm(config.hidden_dim, eps=config.rms_norm_eps)
#         self.attn = MultiHeadAttention(config)
#         self.ln2 = RMSNorm(config.hidden_dim, eps=config.rms_norm_eps)
#         self.ffn = SwiGLUFeedForward(config.hidden_dim, config.intermediate_dim)
    
#     def forward(
#         self, 
#         x: torch.Tensor,
#         attention_mask: torch.Tensor | None = None,
#     ) -> torch.Tensor:
#         x = x + self.attn(self.ln1(x), attention_mask)
#         x = x + self.ffn(self.ln2(x))
#         return x


# class BaselineTransformer(nn.Module):
#     """
#     Transformer language model.
#     """
    
#     def __init__(self, config: TransformerConfig):
#         super().__init__()
#         self.config = config

#         self.embedding = nn.Embedding(config.vocab_size, config.hidden_dim)
        
#         self.layers = nn.ModuleList([
#             TransformerBlock(config) for _ in range(config.num_layers)
#         ])

#         self.ln_f = RMSNorm(config.hidden_dim, eps=config.rms_norm_eps)
#         self.lm_head = nn.Linear(config.hidden_dim, config.vocab_size, bias=False)

#         self.apply(self._init_weights)
    
#     def _init_weights(self, module):
#         if isinstance(module, nn.Linear):
#             torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
#             if module.bias is not None:
#                 torch.nn.init.zeros_(module.bias)
#         elif isinstance(module, nn.Embedding):
#             torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
#     def forward(
#         self, 
#         input_ids: torch.Tensor,
#         attention_mask: torch.Tensor | None = None,
#     ) -> torch.Tensor:
#         """
#         Args:
#             input_ids: (B, S) token indices
#             attention_mask: optional attention mask
            
#         Returns:
#             logits: (B, S, vocab_size)
#         """
#         B, S = input_ids.shape
#         x = self.embedding(input_ids)

#         for layer in self.layers:
#             x = layer(x, attention_mask)

#         x = self.ln_f(x)
#         logits = self.lm_head(x)
#         return logits.float()

#     def compute_loss(
#         self, 
#         logits: torch.Tensor, 
#         labels: torch.Tensor,
#     ) -> torch.Tensor:
#         """
#         Compute cross-entropy loss for language modeling.
#         """
#         return cross_entropy_loss(logits, labels)


Writing model/transformer.py


# `/tests`

In [27]:
# !mkdir tests

In [28]:
# %%writefile tests/conftest.py
# """
# Shared pytest fixtures and utilities for all tests.
# """

# import os
# import sys

# import pytest
# import torch
# from torch.autograd.graph import saved_tensors_hooks

# sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

# SEED = 42


# @pytest.fixture(autouse=True)
# def seed():
#     """Set random seeds for reproducibility."""
#     torch.manual_seed(SEED)
#     torch.cuda.manual_seed(SEED)


# @pytest.fixture
# def device():
#     """CUDA device fixture."""
#     return torch.device("cuda")


# def bytes_to_mb(num_bytes: int) -> float:
#     """Convert bytes to megabytes."""
#     return num_bytes / (1024 ** 2)


# def measure_saved_tensors_bytes(fn, exclude_tensors=None) -> int:
#     """Measure total bytes of new tensors saved for backward pass.
    
#     Args:
#         fn: Function to measure
#         exclude_tensors: List of tensors to exclude (e.g., model parameters)
#     """
#     exclude_ptrs = {t.data_ptr() for t in (exclude_tensors or [])}
#     total = 0

#     def pack(t: torch.Tensor):
#         nonlocal total
#         if t.data_ptr() not in exclude_ptrs:
#             total += t.numel() * t.element_size()
#         return t

#     def unpack(t):
#         return t

#     with saved_tensors_hooks(pack, unpack):
#         fn()
#     return total


# def measure_peak_memory(fn) -> int:
#     """Measure peak GPU memory during fn execution."""
#     torch.cuda.synchronize()
#     torch.cuda.empty_cache()
#     torch.cuda.reset_peak_memory_stats()

#     fn()

#     torch.cuda.synchronize()
#     return torch.cuda.max_memory_allocated()


Writing tests/conftest.py


In [29]:
# %%writefile tests/test_loss.py
# """
# Tests for Cross Entropy Loss correctness and memory consumption.
# """

# import pytest
# import torch

# from conftest import bytes_to_mb, measure_saved_tensors_bytes
# from efficient_model.loss import CrossEntropyLoss as EfficientCrossEntropyLoss
# from model.loss import cross_entropy_loss as baseline_cross_entropy_loss


# HIDDEN_DIM = 256
# VOCAB_SIZE = 1024
# BATCH_SIZE = 4
# SEQ_LEN = 1024


# class TestCrossEntropyLossCorrectness:
#     """Test correctness of efficient CrossEntropyLoss implementation."""

#     @pytest.mark.parametrize("dtype,atol,rtol", [
#         (torch.bfloat16, 1e-2, 1e-2),
#     ])
#     def test_forward(self, device, dtype, atol, rtol):
#         """Test that efficient CrossEntropyLoss forward matches baseline."""
#         hidden_states = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM, device=device, dtype=dtype)
#         lm_head_weight = torch.randn(VOCAB_SIZE, HIDDEN_DIM, device=device, dtype=dtype)
#         labels = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN), device=device)

#         logits = hidden_states @ lm_head_weight.T
#         loss_baseline = baseline_cross_entropy_loss(logits, labels)

#         efficient_loss_fn = EfficientCrossEntropyLoss()
#         loss_efficient = efficient_loss_fn(hidden_states, lm_head_weight, labels)
        
#         diff = (loss_baseline - loss_efficient).abs().item()
#         assert torch.allclose(loss_baseline, loss_efficient, atol=atol, rtol=rtol), \
#             f"Forward loss mismatch! Baseline: {loss_baseline.item():.6f}, Efficient: {loss_efficient.item():.6f}, Diff: {diff:.6f}"

#     @pytest.mark.parametrize("dtype,atol,rtol", [
#         (torch.bfloat16, 1e-1, 1e-2),
#     ])
#     def test_backward(self, device, dtype, atol, rtol):
#         """Test that efficient CrossEntropyLoss backward matches baseline."""
#         hidden_baseline = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM, device=device, dtype=dtype, requires_grad=True)
#         lm_head_baseline = torch.randn(VOCAB_SIZE, HIDDEN_DIM, device=device, dtype=dtype, requires_grad=True)
#         labels = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN), device=device)

#         hidden_efficient = hidden_baseline.clone().detach().requires_grad_(True)
#         lm_head_efficient = lm_head_baseline.clone().detach().requires_grad_(True)

#         logits = hidden_baseline @ lm_head_baseline.T
#         loss_baseline = baseline_cross_entropy_loss(logits, labels)
#         loss_baseline.backward()

#         efficient_loss_fn = EfficientCrossEntropyLoss()
#         loss_efficient = efficient_loss_fn(hidden_efficient, lm_head_efficient, labels)
#         loss_efficient.backward()

#         grad_hidden_diff = (hidden_baseline.grad - hidden_efficient.grad.to(dtype)).abs().max().item()
#         assert torch.allclose(hidden_baseline.grad, hidden_efficient.grad.to(dtype), atol=atol, rtol=rtol), \
#             f"Backward grad_hidden mismatch! Max diff: {grad_hidden_diff}"

#         grad_weight_diff = (lm_head_baseline.grad - lm_head_efficient.grad.to(dtype)).abs().max().item()
#         assert torch.allclose(lm_head_baseline.grad, lm_head_efficient.grad.to(dtype), atol=atol, rtol=rtol), \
#             f"Backward grad_lm_head mismatch! Max diff: {grad_weight_diff}"


# class TestCrossEntropyLossMemory:
#     """Test memory efficiency of efficient CrossEntropyLoss implementation."""

#     def test_saved_tensors_reduction(self, device):
#         """Test that efficient implementation reduces saved tensors."""
#         dtype = torch.bfloat16
        
#         hidden_states = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM, device=device, dtype=dtype, requires_grad=True)
#         lm_head_weight = torch.randn(VOCAB_SIZE, HIDDEN_DIM, device=device, dtype=dtype, requires_grad=True)
#         labels = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN), device=device)
        
#         efficient_loss_fn = EfficientCrossEntropyLoss()

#         def baseline_fwbw_step():
#             if hidden_states.grad is not None:
#                 hidden_states.grad = None
#             if lm_head_weight.grad is not None:
#                 lm_head_weight.grad = None
#             logits = hidden_states @ lm_head_weight.T
#             loss = baseline_cross_entropy_loss(logits, labels)
#             loss.backward()

#         def efficient_fwbw_step():
#             if hidden_states.grad is not None:
#                 hidden_states.grad = None
#             if lm_head_weight.grad is not None:
#                 lm_head_weight.grad = None
#             loss = efficient_loss_fn(hidden_states, lm_head_weight, labels)
#             loss.backward()

#         baseline_saved = measure_saved_tensors_bytes(baseline_fwbw_step)
#         efficient_saved = measure_saved_tensors_bytes(efficient_fwbw_step)

#         reduction = baseline_saved / efficient_saved if efficient_saved > 0 else float("inf")

#         assert reduction >= 6.2, \
#             f"Efficient should use less memory! Baseline: {bytes_to_mb(baseline_saved):.2f} MB, " \
#             f"Efficient: {bytes_to_mb(efficient_saved):.2f} MB, Reduction: {reduction:.2f}x"

Writing tests/test_loss.py


In [30]:
# %%writefile tests/test_attention.py

# """
# Tests for Multi-Head Attention correctness and memory consumption.
# """

# import pytest
# import torch

# from conftest import bytes_to_mb, measure_saved_tensors_bytes, measure_peak_memory
# from config import TransformerConfig
# from efficient_model.attention import MultiHeadAttention as EfficientAttention
# from model.attention import MultiHeadAttention as BaselineAttention


# HIDDEN_DIM = 256
# NUM_HEADS = 8
# BATCH_SIZE = 4
# SEQ_LEN = 1024


# @pytest.fixture
# def config():
#     """Create a test config with smaller dimensions."""
#     return TransformerConfig(
#         hidden_dim=HIDDEN_DIM,
#         num_heads=NUM_HEADS,
#         max_seq_len=SEQ_LEN * 2,
#         dropout=0.0,
#     )


# @pytest.fixture
# def models(device, config):
#     """Create baseline and efficient models with shared weights."""
#     def _create_models(dtype):
#         baseline = BaselineAttention(config).to(device=device, dtype=dtype)
#         efficient = EfficientAttention(config).to(device=device, dtype=dtype)

#         with torch.no_grad():
#             efficient.qkv_proj.weight.data.copy_(
#                 torch.cat([
#                     baseline.q_proj.weight.data,
#                     baseline.k_proj.weight.data,
#                     baseline.v_proj.weight.data,
#                 ], dim=0)
#             )
#             efficient.out_proj.weight.data.copy_(baseline.out_proj.weight.data)

#         baseline.eval()
#         efficient.eval()

#         return baseline, efficient
#     return _create_models


# class TestAttentionCorrectness:
#     """Test correctness of efficient attention implementation."""

#     @pytest.mark.parametrize("dtype,atol,rtol", [
#         (torch.bfloat16, 5e-2, 5e-2),
#     ])
#     def test_forward(self, device, models, dtype, atol, rtol):
#         """Test that efficient attention forward matches baseline."""
#         baseline, efficient = models(dtype)

#         x = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM, device=device, dtype=dtype)

#         with torch.no_grad():
#             y_baseline = baseline(x)
#             y_efficient = efficient(x)

#         max_diff = (y_baseline - y_efficient).abs().max().item()
#         assert torch.allclose(y_baseline, y_efficient, atol=atol, rtol=rtol), \
#             f"Forward mismatch! Max diff: {max_diff}"

#     @pytest.mark.parametrize("dtype,atol,rtol", [
#         (torch.bfloat16, 9.5e-2, 9.5e-2),
#     ])
#     def test_backward(self, device, models, dtype, atol, rtol):
#         """Test that efficient attention backward matches baseline."""
#         baseline, efficient = models(dtype)
#         baseline.train()
#         efficient.train()

#         x_baseline = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM, device=device, dtype=dtype)
#         x_baseline.requires_grad = True
#         x_efficient = x_baseline.clone().detach().requires_grad_(True)

#         y_baseline = baseline(x_baseline)
#         y_efficient = efficient(x_efficient)

#         grad_output = torch.randn_like(y_baseline)
#         y_baseline.backward(grad_output)
#         y_efficient.backward(grad_output)

#         # grad_x_diff = (x_baseline.grad - x_efficient.grad).abs().max().item()
#         # assert torch.allclose(x_baseline.grad, x_efficient.grad, atol=atol, rtol=rtol), \
#         #     f"Backward grad_x mismatch! Max diff: {grad_x_diff}"

#         baseline_qkv_grad = torch.cat([
#             baseline.q_proj.weight.grad,
#             baseline.k_proj.weight.grad,
#             baseline.v_proj.weight.grad,
#         ], dim=0)
#         grad_qkv_diff = (baseline_qkv_grad - efficient.qkv_proj.weight.grad).abs().max().item()
#         assert torch.allclose(baseline_qkv_grad, efficient.qkv_proj.weight.grad, atol=atol, rtol=rtol), \
#             f"Backward grad_qkv mismatch! Max diff: {grad_qkv_diff}"

#         grad_out_diff = (baseline.out_proj.weight.grad - efficient.out_proj.weight.grad).abs().max().item()
#         assert torch.allclose(baseline.out_proj.weight.grad, efficient.out_proj.weight.grad, atol=atol, rtol=rtol), \
#             f"Backward grad_out_proj mismatch! Max diff: {grad_out_diff}"


# class TestAttentionMemory:
#     """Test memory efficiency of efficient attention implementation."""

#     def test_saved_tensors_reduction(self, device, models):
#         """Test that efficient implementation reduces saved tensors."""
#         dtype = torch.bfloat16
#         baseline, efficient = models(dtype)
#         baseline.train()
#         efficient.train()

#         x_fwbw = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM, device=device, dtype=dtype, requires_grad=True)

#         def baseline_fwbw_step():
#             baseline.zero_grad(set_to_none=True)
#             if x_fwbw.grad is not None:
#                 x_fwbw.grad = None
#             y = baseline(x_fwbw)
#             y.sum().backward()

#         def efficient_fwbw_step():
#             efficient.zero_grad(set_to_none=True)
#             if x_fwbw.grad is not None:
#                 x_fwbw.grad = None
#             y = efficient(x_fwbw)
#             y.sum().backward()

#         baseline_saved = measure_saved_tensors_bytes(baseline_fwbw_step, exclude_tensors=list(baseline.parameters()))
#         efficient_saved = measure_saved_tensors_bytes(efficient_fwbw_step, exclude_tensors=list(efficient.parameters()))

#         reduction = baseline_saved / efficient_saved if efficient_saved > 0 else float("inf")
#         min_reduction = 6.0

#         assert reduction >= min_reduction, \
#             f"Expected at least {min_reduction}x reduction, got {reduction:.2f}x " \
#             f"(Baseline: {bytes_to_mb(baseline_saved):.2f} MB, Efficient: {bytes_to_mb(efficient_saved):.2f} MB)"

#     def test_fwbw_peak_memory(self, device, models):
#         """Test that efficient forward+backward uses less peak memory."""
#         dtype = torch.bfloat16
#         baseline, efficient = models(dtype)
#         baseline.train()
#         efficient.train()

#         def fwbw_step(model, x):
#             y = model(x)
#             y.sum().backward()

#         x_baseline = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM, device=device, dtype=dtype, requires_grad=True)
#         baseline_peak = measure_peak_memory(
#             lambda: fwbw_step(baseline, x_baseline)
#         )

#         del x_baseline
#         baseline.zero_grad(set_to_none=True)
#         torch.cuda.empty_cache()

#         x_efficient = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM, device=device, dtype=dtype, requires_grad=True)
#         efficient_peak = measure_peak_memory(
#             lambda: fwbw_step(efficient, x_efficient)
#         )

#         diff_mb = bytes_to_mb(baseline_peak) - bytes_to_mb(efficient_peak)
#         min_diff_mb = 50

#         assert diff_mb >= min_diff_mb, \
#             f"Expected at least {min_diff_mb} MB reduction, got {diff_mb:.2f} MB " \
#             f"(Baseline: {bytes_to_mb(baseline_peak):.2f} MB, Efficient: {bytes_to_mb(efficient_peak):.2f} MB)"


Writing tests/test_attention.py


In [31]:
# %%writefile tests/test_rmsnorm.py
# """
# Tests for RMSNorm correctness and memory consumption.
# """

# import pytest
# import torch

# from conftest import bytes_to_mb, measure_saved_tensors_bytes
# from efficient_model.norm import RMSNorm as EfficientRMSNorm
# from model.norm import RMSNorm as BaselineRMSNorm


# HIDDEN_DIM = 256
# BATCH_SIZE = 4
# SEQ_LEN = 1024


# @pytest.fixture
# def models(device):
#     """Create baseline and efficient models with shared weights."""
#     def _create_models(dtype):
#         baseline = BaselineRMSNorm(HIDDEN_DIM).to(device=device, dtype=dtype)
#         efficient = EfficientRMSNorm(HIDDEN_DIM).to(device=device, dtype=dtype)
#         efficient.weight.data.copy_(baseline.weight.data)
#         return baseline, efficient
#     return _create_models


# class TestRMSNormCorrectness:
#     """Test correctness of efficient RMSNorm implementation."""

#     @pytest.mark.parametrize("dtype,atol,rtol", [
#         (torch.bfloat16, 2e-2, 2e-2),
#     ])
#     def test_forward(self, device, models, dtype, atol, rtol):
#         """Test that efficient RMSNorm forward matches baseline."""
#         baseline, efficient = models(dtype)

#         x = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM, device=device, dtype=dtype)

#         y_baseline = baseline(x)
#         y_efficient = efficient(x)

#         max_diff = (y_baseline - y_efficient).abs().max().item()
#         assert torch.allclose(y_baseline, y_efficient, atol=atol, rtol=rtol), \
#             f"Forward mismatch! Max diff: {max_diff}"

#     @pytest.mark.parametrize("dtype,rtol", [
#         (torch.bfloat16, 2e-2),
#     ])
#     def test_backward(self, device, models, dtype, rtol):
#         """Test that efficient RMSNorm backward matches baseline."""
#         baseline, efficient = models(dtype)

#         x_baseline = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM, device=device, dtype=dtype)
#         x_baseline.requires_grad = True
#         x_efficient = x_baseline.clone().detach().requires_grad_(True)

#         y_baseline = baseline(x_baseline)
#         y_efficient = efficient(x_efficient)

#         grad_output = torch.randn_like(y_baseline)
#         y_baseline.backward(grad_output)
#         y_efficient.backward(grad_output)

#         grad_x_diff = (x_baseline.grad - x_efficient.grad).abs().max().item()
#         assert torch.allclose(x_baseline.grad, x_efficient.grad, atol=2e-2, rtol=rtol), \
#             f"Backward grad_x mismatch! Max diff: {grad_x_diff}"

#         grad_w_diff = (baseline.weight.grad - efficient.weight.grad).abs().max().item()
#         assert torch.allclose(baseline.weight.grad, efficient.weight.grad, atol=5e-1, rtol=rtol), \
#             f"Backward grad_weight mismatch! Max diff: {grad_w_diff}"


# class TestRMSNormMemory:
#     """Test memory efficiency of efficient RMSNorm implementation."""

#     def test_saved_tensors_reduction(self, device, models):
#         """Test that efficient implementation reduces saved tensors by at least 7.9x."""
#         dtype = torch.bfloat16
#         baseline, efficient = models(dtype)

#         x_fwbw = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM, device=device, dtype=dtype, requires_grad=True)

#         def baseline_fwbw_step():
#             baseline.zero_grad(set_to_none=True)
#             if x_fwbw.grad is not None:
#                 x_fwbw.grad = None
#             y = baseline(x_fwbw)
#             y.sum().backward()

#         def efficient_fwbw_step():
#             efficient.zero_grad(set_to_none=True)
#             if x_fwbw.grad is not None:
#                 x_fwbw.grad = None
#             y = efficient(x_fwbw)
#             y.sum().backward()

#         baseline_saved = measure_saved_tensors_bytes(baseline_fwbw_step, exclude_tensors=list(baseline.parameters()))
#         efficient_saved = measure_saved_tensors_bytes(efficient_fwbw_step, exclude_tensors=list(efficient.parameters()))

#         reduction = baseline_saved / efficient_saved if efficient_saved > 0 else float("inf")
#         min_reduction = 7.9

#         assert reduction >= min_reduction, \
#             f"Expected at least {min_reduction}x reduction, got {reduction:.2f}x " \
#             f"(Baseline: {bytes_to_mb(baseline_saved):.2f} MB, Efficient: {bytes_to_mb(efficient_saved):.2f} MB)"


Writing tests/test_rmsnorm.py


In [32]:
# %%writefile tests/test_swiglu.py
# """
# Tests for SwiGLU Feed-Forward Network correctness and memory consumption.
# """

# import pytest
# import torch

# from conftest import bytes_to_mb, measure_saved_tensors_bytes, measure_peak_memory
# from efficient_model.swiglu import SwiGLUFeedForward as EfficientSwiGLU
# from model.swiglu import SwiGLUFeedForward as BaselineSwiGLU


# HIDDEN_DIM = 256
# INTERMEDIATE_DIM = 512
# BATCH_SIZE = 4
# SEQ_LEN = 1024


# @pytest.fixture
# def models(device):
#     """Create baseline and efficient models with shared weights."""
#     def _create_models(dtype):
#         baseline = BaselineSwiGLU(HIDDEN_DIM, INTERMEDIATE_DIM).to(device=device, dtype=dtype)
#         efficient = EfficientSwiGLU(HIDDEN_DIM, INTERMEDIATE_DIM).to(device=device, dtype=dtype)

#         efficient.gate_proj.weight.data.copy_(baseline.gate_proj.weight.data)
#         efficient.up_proj.weight.data.copy_(baseline.up_proj.weight.data)
#         efficient.down_proj.weight.data.copy_(baseline.down_proj.weight.data)

#         return baseline, efficient
#     return _create_models


# class TestSwiGLUCorrectness:
#     """Test correctness of efficient SwiGLU implementation."""

#     @pytest.mark.parametrize("dtype,atol,rtol", [
#         (torch.float32, 5e-5, 5e-5),
#     ])
#     def test_forward(self, device, models, dtype, atol, rtol):
#         """Test that efficient SwiGLU forward matches baseline."""
#         baseline, efficient = models(dtype)

#         x = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM, device=device, dtype=dtype)

#         y_baseline = baseline(x)
#         y_efficient = efficient(x)

#         max_diff = (y_baseline - y_efficient).abs().max().item()
#         assert torch.allclose(y_baseline, y_efficient, atol=atol, rtol=rtol), \
#             f"Forward mismatch! Max diff: {max_diff}"

#     @pytest.mark.parametrize("dtype,atol,rtol", [
#         (torch.float32, 5e-5, 5e-5),
#     ])
#     def test_backward(self, device, models, dtype, atol, rtol):
#         """Test that efficient SwiGLU backward matches baseline."""
#         baseline, efficient = models(dtype)

#         x_baseline = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM, device=device, dtype=dtype)
#         x_baseline.requires_grad = True
#         x_efficient = x_baseline.clone().detach().requires_grad_(True)

#         y_baseline = baseline(x_baseline)
#         y_efficient = efficient(x_efficient)

#         grad_output = torch.randn_like(y_baseline)
#         y_baseline.backward(grad_output)
#         y_efficient.backward(grad_output)

#         grad_x_diff = (x_baseline.grad - x_efficient.grad).abs().max().item()
#         assert torch.allclose(x_baseline.grad, x_efficient.grad, atol=atol, rtol=rtol), \
#             f"Backward grad_x mismatch! Max diff: {grad_x_diff}"

#         grad_gate_diff = (baseline.gate_proj.weight.grad - efficient.gate_proj.weight.grad).abs().max().item()
#         assert torch.allclose(baseline.gate_proj.weight.grad, efficient.gate_proj.weight.grad, atol=atol, rtol=rtol), \
#             f"Backward grad_gate_proj mismatch! Max diff: {grad_gate_diff}"

#         grad_up_diff = (baseline.up_proj.weight.grad - efficient.up_proj.weight.grad).abs().max().item()
#         assert torch.allclose(baseline.up_proj.weight.grad, efficient.up_proj.weight.grad, atol=atol, rtol=rtol), \
#             f"Backward grad_up_proj mismatch! Max diff: {grad_up_diff}"

#         grad_down_diff = (baseline.down_proj.weight.grad - efficient.down_proj.weight.grad).abs().max().item()
#         assert torch.allclose(baseline.down_proj.weight.grad, efficient.down_proj.weight.grad, atol=atol, rtol=rtol), \
#             f"Backward grad_down_proj mismatch! Max diff: {grad_down_diff}"


# class TestSwiGLUMemory:
#     """Test memory efficiency of efficient SwiGLU implementation."""

#     def test_saved_tensors_reduction(self, device, models):
#         """Test that efficient implementation reduces saved tensors by at least 3.4x."""
#         dtype = torch.float32
#         baseline, efficient = models(dtype)

#         x_fwbw = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM, device=device, dtype=dtype, requires_grad=True)

#         def baseline_fwbw_step():
#             baseline.zero_grad(set_to_none=True)
#             if x_fwbw.grad is not None:
#                 x_fwbw.grad = None
#             y = baseline(x_fwbw)
#             y.sum().backward()

#         def efficient_fwbw_step():
#             efficient.zero_grad(set_to_none=True)
#             if x_fwbw.grad is not None:
#                 x_fwbw.grad = None
#             y = efficient(x_fwbw)
#             y.sum().backward()

#         baseline_saved = measure_saved_tensors_bytes(baseline_fwbw_step, exclude_tensors=list(baseline.parameters()))
#         efficient_saved = measure_saved_tensors_bytes(efficient_fwbw_step, exclude_tensors=list(efficient.parameters()))

#         reduction = baseline_saved / efficient_saved if efficient_saved > 0 else float("inf")
#         min_reduction = 3.6

#         assert reduction >= min_reduction, \
#             f"Expected at least {min_reduction}x reduction, got {reduction:.2f}x " \
#             f"(Baseline: {bytes_to_mb(baseline_saved):.2f} MB, Efficient: {bytes_to_mb(efficient_saved):.2f} MB)"

#     def test_fwbw_peak_memory(self, device, models):
#         """Test that efficient forward+backward uses less peak memory."""
#         dtype = torch.bfloat16
#         baseline, efficient = models(dtype)

#         def fwbw_step(model, x):
#             y = model(x)
#             y.sum().backward()

#         x_baseline = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM, device=device, dtype=dtype, requires_grad=True)
#         baseline_peak = measure_peak_memory(
#             lambda: fwbw_step(baseline, x_baseline)
#         )

#         del x_baseline
#         baseline.zero_grad(set_to_none=True)
#         torch.cuda.empty_cache()

#         x_efficient = torch.randn(BATCH_SIZE, SEQ_LEN, HIDDEN_DIM, device=device, dtype=dtype, requires_grad=True)
#         efficient_peak = measure_peak_memory(
#             lambda: fwbw_step(efficient, x_efficient)
#         )

#         diff_mb = bytes_to_mb(baseline_peak) - bytes_to_mb(efficient_peak)
#         min_diff_mb = 20

#         assert diff_mb >= min_diff_mb, \
#             f"Expected at least {min_diff_mb} MB reduction, got {diff_mb:.2f} MB " \
#             f"(Baseline: {bytes_to_mb(baseline_peak):.2f} MB, Efficient: {bytes_to_mb(efficient_peak):.2f} MB)"


Writing tests/test_swiglu.py


In [33]:
# %%writefile tests/test_e2e.py
# """
# End-to-end tests for the full Transformer model.
# """

# import pytest
# import torch

# from conftest import bytes_to_mb, measure_peak_memory
# from config import TransformerConfig
# from model.transformer import BaselineTransformer
# from model.loss import cross_entropy_loss
# from efficient_model.transformer import EfficientTransformer


# VOCAB_SIZE = 16000
# HIDDEN_DIM = 512
# NUM_HEADS = 8
# NUM_LAYERS = 6
# INTERMEDIATE_DIM = 1024
# MAX_SEQ_LEN = 4096
# BATCH_SIZE = 2
# SEQ_LEN = 4096


# @pytest.fixture
# def config():
#     """Create a small test config."""
#     return TransformerConfig(
#         vocab_size=VOCAB_SIZE,
#         hidden_dim=HIDDEN_DIM,
#         num_heads=NUM_HEADS,
#         num_layers=NUM_LAYERS,
#         intermediate_dim=INTERMEDIATE_DIM,
#         max_seq_len=MAX_SEQ_LEN,
#         dropout=0.0,
#     )


# @pytest.fixture
# def models(device, config):
#     """Create baseline and efficient models with shared weights."""
#     def _create_models(dtype):
#         baseline = BaselineTransformer(config).to(device=device, dtype=dtype)
#         efficient = EfficientTransformer(config).to(device=device, dtype=dtype)

#         with torch.no_grad():
#             efficient.embedding.weight.data.copy_(baseline.embedding.weight.data)

#             for b_layer, e_layer in zip(baseline.layers, efficient.layers):
#                 e_layer.ln1.weight.data.copy_(b_layer.ln1.weight.data)
#                 e_layer.ln2.weight.data.copy_(b_layer.ln2.weight.data)

#                 e_layer.attn.qkv_proj.weight.data.copy_(
#                     torch.cat([
#                         b_layer.attn.q_proj.weight.data,
#                         b_layer.attn.k_proj.weight.data,
#                         b_layer.attn.v_proj.weight.data,
#                     ], dim=0)
#                 )
#                 e_layer.attn.out_proj.weight.data.copy_(b_layer.attn.out_proj.weight.data)

#                 e_layer.ffn.gate_proj.weight.data.copy_(b_layer.ffn.gate_proj.weight.data)
#                 e_layer.ffn.up_proj.weight.data.copy_(b_layer.ffn.up_proj.weight.data)
#                 e_layer.ffn.down_proj.weight.data.copy_(b_layer.ffn.down_proj.weight.data)

#             efficient.ln_f.weight.data.copy_(baseline.ln_f.weight.data)
#             efficient.lm_head.weight.data.copy_(baseline.lm_head.weight.data)

#         return baseline, efficient
#     return _create_models


# class TestE2ECorrectness:
#     """Test end-to-end correctness of the optimized model."""

#     @pytest.mark.parametrize("dtype,atol,rtol", [
#         (torch.bfloat16, 5e-2, 5e-2),
#     ])
#     def test_forward(self, device, models, dtype, atol, rtol):
#         """Test that efficient model produces same loss as baseline."""
#         baseline, efficient = models(dtype)
#         baseline.eval()
#         efficient.eval()

#         input_ids = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN), device=device)
#         labels = input_ids.clone()

#         with torch.no_grad():
#             logits_baseline = baseline(input_ids)
#             loss_baseline = cross_entropy_loss(logits_baseline, labels)

#             loss_efficient = efficient(input_ids, labels=labels)

#         loss_diff = (loss_baseline - loss_efficient).abs().item()
#         assert torch.allclose(loss_baseline, loss_efficient, atol=atol, rtol=rtol), \
#             f"Forward loss mismatch! Baseline: {loss_baseline.item():.4f}, Efficient: {loss_efficient.item():.4f}, Diff: {loss_diff:.4f}"

#     @pytest.mark.parametrize("dtype,atol,rtol", [
#         (torch.bfloat16, 5e-2, 5e-2),
#     ])
#     def test_backward(self, device, models, dtype, atol, rtol):
#         """Test that efficient model produces same gradients as baseline."""
#         baseline, efficient = models(dtype)
#         baseline.train()
#         efficient.train()

#         input_ids = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN), device=device)
#         labels = input_ids.clone()

#         logits_baseline = baseline(input_ids)
#         loss_baseline = cross_entropy_loss(logits_baseline, labels)
#         loss_baseline.backward()

#         loss_efficient = efficient(input_ids, labels=labels)
#         loss_efficient.backward()

#         emb_grad_diff = (baseline.embedding.weight.grad - efficient.embedding.weight.grad).abs().max().item()
#         assert torch.allclose(baseline.embedding.weight.grad, efficient.embedding.weight.grad, atol=atol, rtol=rtol), \
#             f"Embedding grad mismatch! Max diff: {emb_grad_diff}"

#         lm_grad_diff = (baseline.lm_head.weight.grad - efficient.lm_head.weight.grad).abs().max().item()
#         assert torch.allclose(baseline.lm_head.weight.grad, efficient.lm_head.weight.grad, atol=atol, rtol=rtol), \
#             f"LM head grad mismatch! Max diff: {lm_grad_diff}"


# class TestE2EMemory:
#     """Test memory efficiency of the optimized model."""

#     def test_peak_memory(self, device, models):
#         """Test that efficient model uses less peak memory."""
#         dtype = torch.bfloat16
#         baseline, efficient = models(dtype)
#         baseline.train()
#         efficient.train()

#         input_ids = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN), device=device)
#         labels = input_ids.clone()

#         def baseline_fwbw():
#             baseline.zero_grad(set_to_none=True)
#             logits = baseline(input_ids)
#             loss = cross_entropy_loss(logits, labels)
#             loss.backward()

#         def efficient_fwbw():
#             efficient.zero_grad(set_to_none=True)
#             loss = efficient(input_ids, labels=labels)
#             loss.backward()

#         baseline_peak = measure_peak_memory(baseline_fwbw)

#         baseline.zero_grad(set_to_none=True)
#         torch.cuda.empty_cache()

#         efficient_peak = measure_peak_memory(efficient_fwbw)

#         ratio = baseline_peak / efficient_peak if efficient_peak > 0 else float("inf")
#         min_ratio = 1.2

#         assert ratio >= min_ratio, \
#             f"Expected at least {min_ratio}x memory reduction, got {ratio:.2f}x " \
#             f"(Baseline: {bytes_to_mb(baseline_peak):.2f} MB, Efficient: {bytes_to_mb(efficient_peak):.2f} MB)"


Writing tests/test_e2e.py


In [34]:
# %%writefile tests/test_optimizer.py
# """
# Tests for optimizer step correctness.
# """

# import pytest
# import torch
# import torch.nn as nn

# from optimizer.ademamix import AdEMAMix as AdemamixForloop
# from efficient_optimizer.ademamix import AdEMAMix as AdemamixForeach

# import torch._dynamo as dynamo
# dynamo.config.recompile_limit = 8

# HIDDEN_DIM = 16
# NUM_LAYERS = 3
# NUM_STEPS = 100

# def _build_model(device: torch.device, dtype: torch.dtype) -> nn.Module:
#     torch.manual_seed(0)
#     layers = [nn.Linear(HIDDEN_DIM, HIDDEN_DIM, bias=True) for _ in range(NUM_LAYERS)]
#     return nn.Sequential(*layers).to(device=device, dtype=dtype)


# def _assert_models_close(model_a: nn.Module, model_b: nn.Module, step: int, rtol: float=1e-5, atol: float=1e-6) -> None:
#     a = dict(model_a.named_parameters())
#     b = dict(model_b.named_parameters())
#     assert a.keys() == b.keys()

#     for name in a.keys():
#         pa, pb = a[name].data, b[name].data
#         max_diff = (pa - pb).abs().max().item()
#         assert torch.allclose(pa, pb, atol=atol, rtol=rtol), (
#             f"Param mismatch at step={step}, name={name}, max_diff={max_diff}"
#         )


# def _apply_random_grads(model_a: nn.Module, model_b: nn.Module) -> None:
#     torch.manual_seed(0)
#     a = dict(model_a.named_parameters())
#     b = dict(model_b.named_parameters())
#     for name in a.keys():
#         g = torch.randn_like(a[name].data)
#         a[name].grad = g
#         b[name].grad = g.clone()


# class TestCorrectness:
#     """Test correctness of efficient AdEMAMix implementation."""
#     @pytest.mark.parametrize("dtype,atol,rtol", [
#         (torch.float32, 1e-6, 1e-5),
#     ])
#     def test_steps_match(self, device, dtype, atol, rtol):
#         model_baseline = _build_model(device, dtype)
#         model_efficient = _build_model(device, dtype)

#         opt_baseline = AdemamixForloop(model_baseline.parameters(), lr=1e-2, weight_decay=0.1, alpha_warmup=51, beta3_warmup=51)
#         opt_efficient = AdemamixForeach(model_efficient.parameters(), lr=1e-2, weight_decay=0.1, alpha_warmup=51, beta3_warmup=51)

#         _assert_models_close(model_baseline, model_efficient, step=0, atol=atol, rtol=rtol)

#         for step in range(1, NUM_STEPS + 1):
#             _apply_random_grads(model_baseline, model_efficient)
#             opt_baseline.step()
#             opt_efficient.step()
#             _assert_models_close(model_baseline, model_efficient, step=step, atol=atol, rtol=rtol)


Writing tests/test_optimizer.py


# *testing*

In [9]:
!pytest tests/test_loss.py

=================================================================== test session starts ===================================================================
platform linux -- Python 3.12.13, pytest-9.0.3, pluggy-1.6.0
rootdir: /workspace/efficient-llm-anatomy
plugins: anyio-4.13.0
collected 3 items                                                                                                                                         

tests/test_loss.py ...                                                                                                                              [100%]

==================================================================== warnings summary =====================================================================
../../venv/main/lib/python3.12/site-packages/torch/jit/_script.py:365: 14 warnings
  /venv/main/lib/python3.12/site-packages/torch/jit/_script.py:365: DeprecationWarning: `torch.jit.script_method` is deprecated. Please switch to `torch.compile` or `to

In [10]:
!pytest tests/test_attention.py

=================================================================== test session starts ===================================================================
platform linux -- Python 3.12.13, pytest-9.0.3, pluggy-1.6.0
rootdir: /workspace/efficient-llm-anatomy
plugins: anyio-4.13.0
collected 4 items                                                                                                                                         

tests/test_attention.py ....                                                                                                                        [100%]

==================================================================== warnings summary =====================================================================
../../venv/main/lib/python3.12/site-packages/torch/jit/_script.py:365: 14 warnings
  /venv/main/lib/python3.12/site-packages/torch/jit/_script.py:365: DeprecationWarning: `torch.jit.script_method` is deprecated. Please switch to `torch.compile` or `to

In [11]:
!pytest tests/test_rmsnorm.py

=================================================================== test session starts ===================================================================
platform linux -- Python 3.12.13, pytest-9.0.3, pluggy-1.6.0
rootdir: /workspace/efficient-llm-anatomy
plugins: anyio-4.13.0
collected 3 items                                                                                                                                         

tests/test_rmsnorm.py ...                                                                                                                           [100%]

==================================================================== warnings summary =====================================================================
../../venv/main/lib/python3.12/site-packages/torch/jit/_script.py:365: 14 warnings
  /venv/main/lib/python3.12/site-packages/torch/jit/_script.py:365: DeprecationWarning: `torch.jit.script_method` is deprecated. Please switch to `torch.compile` or `to

In [12]:
!pytest tests/test_swiglu.py

=================================================================== test session starts ===================================================================
platform linux -- Python 3.12.13, pytest-9.0.3, pluggy-1.6.0
rootdir: /workspace/efficient-llm-anatomy
plugins: anyio-4.13.0
collected 4 items                                                                                                                                         

tests/test_swiglu.py ....                                                                                                                           [100%]

==================================================================== warnings summary =====================================================================
../../venv/main/lib/python3.12/site-packages/torch/jit/_script.py:365: 14 warnings
  /venv/main/lib/python3.12/site-packages/torch/jit/_script.py:365: DeprecationWarning: `torch.jit.script_method` is deprecated. Please switch to `torch.compile` or `to

In [13]:
!pytest tests/test_e2e.py

=================================================================== test session starts ===================================================================
platform linux -- Python 3.12.13, pytest-9.0.3, pluggy-1.6.0
rootdir: /workspace/efficient-llm-anatomy
plugins: anyio-4.13.0
collected 3 items                                                                                                                                         

tests/test_e2e.py ...                                                                                                                               [100%]

==================================================================== warnings summary =====================================================================
../../venv/main/lib/python3.12/site-packages/torch/jit/_script.py:365: 14 warnings
  /venv/main/lib/python3.12/site-packages/torch/jit/_script.py:365: DeprecationWarning: `torch.jit.script_method` is deprecated. Please switch to `torch.compile` or `to

In [14]:
!pytest tests/test_optimizer.py

=================================================================== test session starts ===================================================================
platform linux -- Python 3.12.13, pytest-9.0.3, pluggy-1.6.0
rootdir: /workspace/efficient-llm-anatomy
plugins: anyio-4.13.0
collected 1 item                                                                                                                                          

tests/test_optimizer.py .                                                                                                                           [100%]

==================================================================== warnings summary =====================================================================
tests/test_optimizer.py: 14 warnings
  /venv/main/lib/python3.12/site-packages/torch/jit/_script.py:365: DeprecationWarning: `torch.jit.script_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
    warnings.warn(

-- Docs: http

In [16]:
!torchrun --nproc_per_node=1 train.py --num-epochs=1 --batch-size=2

Training with 1 GPU(s)
Device: cuda:0
Model config: TransformerConfig(vocab_size=16000, hidden_dim=512, num_heads=8, num_layers=6, intermediate_dim=1024, max_seq_len=4096, dropout=0.0, rope_theta=10000.0, rms_norm_eps=1e-06)
Model parameters: 32,119,296

Training for 1 epochs (500 steps)
Batch size: 2 x 1 = 2
Sequence length 4096
------------------------------------------------------------
Step    10 | Epoch 1/1 | Loss 9.7834 | LR 1.80e-05 | Tokens/s 21,281
Step    20 | Epoch 1/1 | Loss 9.7816 | LR 3.80e-05 | Tokens/s 25,753
Step    30 | Epoch 1/1 | Loss 9.7785 | LR 5.80e-05 | Tokens/s 25,726
Step    40 | Epoch 1/1 | Loss 9.7730 | LR 7.80e-05 | Tokens/s 25,750
Step    50 | Epoch 1/1 | Loss 9.7662 | LR 9.80e-05 | Tokens/s 25,740
Step    60 | Epoch 1/1 | Loss 9.7592 | LR 9.99e-05 | Tokens/s 25,734
Step    70 | Epoch 1/1 | Loss 9.7523 | LR 9.96e-05 | Tokens/s 25,719
Step    80 | Epoch 1/1 | Loss 9.7463 | LR 9.90e-05 | Tokens/s 25,699
^C
W0503 10:12:18.623000 3804 site-packages/torch/distr

In [19]:
!torchrun --nproc_per_node=1 efficient_train.py --num-epochs=1 --batch-size=32

Training with 1 GPU(s)
Device: cuda:0
Model config: TransformerConfig(vocab_size=16000, hidden_dim=512, num_heads=8, num_layers=6, intermediate_dim=1024, max_seq_len=4096, dropout=0.0, rope_theta=10000.0, rms_norm_eps=1e-06)
Model parameters: 32,119,296

Training for 1 epochs (32 steps)
Batch size: 32 x 1 = 32
Sequence length: 4096
------------------------------------------------------------
Step    10 | Epoch 1/1 | Loss 9.7665 | LR 8.98e-05 | Tokens/s 144,081
Step    20 | Epoch 1/1 | Loss 9.7418 | LR 4.19e-05 | Tokens/s 219,951
Step    30 | Epoch 1/1 | Loss 9.7276 | LR 2.62e-06 | Tokens/s 219,722
Epoch 1 complete | Avg Loss: 9.7257


# Benchmarking optimizer

In [54]:
# import torch
# import torch.nn as nn
# from efficient_optimizer import ademamix
# from importlib import reload
# ademamix = reload(ademamix)


# def test_optimizer_has_only_one_kernel():
#     torch._dynamo.reset()
#     torch._logging.set_logs(output_code=True)
    
#     model = nn.Sequential(
#         nn.Linear(256, 256),
#         nn.ReLU(),
#         nn.Linear(256, 256),
#         nn.ReLU(),
#         nn.Linear(256, 256),
#         nn.ReLU(),
#         nn.Linear(256, 128),
#     ).cuda()
#     optimizer = ademamix.AdEMAMix(model.parameters(), use_foreach_map=False)
#     x = torch.randn(4, 256, device="cuda")
#     loss = model(x).sum()
#     loss.backward()
#     optimizer.step()
    
#     torch._logging.set_logs(output_code=False)

In [55]:
# test_optimizer_has_only_one_kernel() # make sure you see only one triton kernel here

In [56]:
# import time
# import gc

# def build_model(hidden=512, layers=24, device="cuda", dtype=torch.float32):
#     """Simple MLP to benchmark optimizer step time."""
#     modules = []
#     for _ in range(layers):
#         modules.append(nn.Linear(hidden, hidden, bias=True))
#         modules.append(nn.ReLU())
#     model = nn.Sequential(*modules).to(device=device, dtype=dtype)
#     return model


# def fake_grads(model):
#     """Set random gradients on all parameters."""
#     for p in model.parameters():
#         p.grad = torch.randn_like(p)


# def benchmark_optimizer(opt, model, warmup_steps=20, bench_steps=100):
#     """Returns median step time in ms and peak memory delta in MB."""
#     fake_grads(model)

#     for _ in range(warmup_steps):
#         opt.step()

#     torch.cuda.synchronize()
#     torch.cuda.reset_peak_memory_stats()
#     mem_before = torch.cuda.memory_allocated()

#     start = time.perf_counter()
#     for _ in range(bench_steps):
#         opt.step()
#     torch.cuda.synchronize()
#     elapsed = time.perf_counter() - start

#     mem_peak = torch.cuda.max_memory_allocated()
#     mem_delta_mb = (mem_peak - mem_before) / 1024 / 1024

#     ms_per_step = (elapsed / bench_steps) * 1000
#     return ms_per_step, mem_delta_mb


# def get_optimizers_summary():
#     device = "cuda"
#     dtype = torch.bfloat16
#     hidden = 256
#     layers = 128 # i set it huge!

#     configs = [
#         ("AdamW", None),
#         ("AdEMAMix (foreach)", False),
#         ("AdEMAMix (foreach_map)", True),
#     ]

#     results = []

#     for name, use_map in configs:
#         gc.collect()
#         torch.cuda.empty_cache()
#         torch.cuda.reset_peak_memory_stats()
#         torch._dynamo.reset()

#         model = build_model(hidden, layers, device, dtype)
#         fake_grads(model)

#         if name == "AdamW":
#             opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
#         else:
#             opt = ademamix.AdEMAMix(
#                 model.parameters(), lr=1e-3, weight_decay=0.1,
#                 alpha_warmup=51, beta3_warmup=51,
#                 use_foreach_map=use_map,
#             )

#         ms, mem = benchmark_optimizer(opt, model)
#         results.append((name, ms, mem))
#         del model, opt

#     print("\n--- Summary ---")
#     for name, ms, mem in results:
#         print(f"{name:30s}  {ms:.3f} ms/step  {mem:.1f} MB peak delta")

In [57]:
# get_optimizers_summary()


--- Summary ---
AdamW                           1.529 ms/step  16.1 MB peak delta
AdEMAMix (foreach)              1.830 ms/step  0.0 MB peak delta
AdEMAMix (foreach_map)          4.196 ms/step  64.3 MB peak delta


# Benchmark & Calculator

In [58]:
# %%writefile layer_bench.py
# import torch
# import time
# import importlib
# import sys

# from config import TransformerConfig
# from model import attention as base_attn, swiglu as base_swiglu, norm as base_norm
# from model.transformer import BaselineTransformer
# from efficient_model import attention as eff_attn, swiglu as eff_swiglu, norm as eff_norm
# from efficient_model.transformer import EfficientTransformer
# from efficient_optimizer.ademamix import AdEMAMix

# import calculators.base, calculators.baseline_calculator, calculators.efficient_calculator
# for m in [calculators.base, calculators.baseline_calculator, calculators.efficient_calculator]:
#     importlib.reload(m)
# from calculators.base import GPUSpec, ModelConfig, TrainingConfig
# from calculators.baseline_calculator import BaselineCalculator
# from calculators.efficient_calculator import EfficientCalculator

# from torch.utils.flop_counter import FlopCounterMode
# from torch.autograd.graph import saved_tensors_hooks


# def bytes_to_mb(num_bytes: int) -> float:
#     return num_bytes / (1024 ** 2)

# def measure_saved_tensors_bytes(fn, exclude_tensors=None):
#     exclude_ptrs = {t.data_ptr() for t in (exclude_tensors or [])}
#     seen_ptrs = set()
#     total = 0
#     def pack(t):
#         nonlocal total
#         ptr = t.data_ptr()
#         if ptr not in exclude_ptrs and ptr not in seen_ptrs:
#             seen_ptrs.add(ptr)
#             total += t.numel() * t.element_size()
#         return t
#     with saved_tensors_hooks(pack, lambda t: t):
#         fn()
#     return total

# def measure_peak_memory(fn) -> int:
#     torch.cuda.synchronize()
#     torch.cuda.empty_cache()
#     torch.cuda.reset_peak_memory_stats()
#     fn()
#     torch.cuda.synchronize()
#     return torch.cuda.max_memory_reserved() # not allocated

# def bench_time(fn, warmup=3, iters=10, with_bwd=False):
#     for _ in range(warmup):
#         fn().sum().backward() if with_bwd else fn()
#         torch.cuda.synchronize()
#     times = []
#     torch.cuda.synchronize()
#     t0 = time.perf_counter()
#     for _ in range(iters):    
#         fn().sum().backward() if with_bwd else fn()
#     torch.cuda.synchronize()
#     return ((time.perf_counter() - t0) * 1000) / iters


# def make_configs(hidden_dim, num_heads, batch_size, seq_len):
#     config = TransformerConfig(
#         hidden_dim=hidden_dim, num_heads=num_heads,
#         max_seq_len=seq_len * 2, dropout=0.0,
#     )
#     mc = ModelConfig(
#         vocab_size=config.vocab_size,
#         hidden_dim=config.hidden_dim,
#         num_heads=config.num_heads,
#         num_layers=config.num_layers,
#         intermediate_dim=config.intermediate_dim,
#         max_seq_len=config.max_seq_len,
#     )
#     tc = TrainingConfig(batch_size=batch_size, seq_len=seq_len, num_gpus=1)
#     return config, mc, tc
    
# def calc_pred_fwd_flops(calc, mc):
#     flops = 0
#     for _ in range(mc.num_layers):
#         flops += calc._attention_breakdown()[0]
#         flops += calc._mlp_breakdown()[0]
#         flops += 2 * calc._rms_norm_breakdown()[0]
#     flops += calc._embedding_breakdown()[0]
#     flops += calc._lm_head_breakdown()[0]
#     flops += calc._loss_breakdown()[0]
#     return flops

# def calc_pred_fwd_bwd_flops(calc, mc):
#     pred_fwd_flops = calc_pred_fwd_flops(calc, mc)
#     loss_flops = calc._loss_breakdown()[0]
#     return (pred_fwd_flops - loss_flops) * 3 + loss_flops
    

# def tflops(flops, time_ms):
#     if time_ms <= 0:
#         return 0.0
#     return flops / (time_ms / 1000) / 1e12


# def bench_layers(bench_time_fn, measure_mem_fn, config, mc, tc):
#     components = [
#         ("Attention",
#          base_attn.MultiHeadAttention(config), eff_attn.MultiHeadAttention(config),
#          "time_attention_ms", "attention_saved_tensors_bytes", "_attention_breakdown"),
#         ("RMSNorm",
#          base_norm.RMSNorm(config.hidden_dim), eff_norm.RMSNorm(config.hidden_dim),
#          "time_rms_norm_ms", "norm_saved_tensors_bytes", "_rms_norm_breakdown"),
#         ("SwiGLU",
#          base_swiglu.SwiGLUFeedForward(config.hidden_dim, config.intermediate_dim),
#          eff_swiglu.SwiGLUFeedForward(config.hidden_dim, config.intermediate_dim),
#          "time_mlp_ms", "mlp_saved_tensors_bytes", "_mlp_breakdown"),
#     ]

#     header = (f"{'Component':<12} {'Type':<12} {'Time ms':>10} {'Pred ms':>10} "
#           f"{'Mem MB':>10} {'Pred MB':>10} {'Torch GF':>10} {'Calc GF':>10} {'TFLOP/s':>10}")
#     print(f"\n{'='*len(header)}")
#     print(f"  Per-Layer  B={tc.batch_size} S={tc.seq_len} H={config.hidden_dim}")
#     print(f"{'='*len(header)}")
#     print(header)
#     print("-" * len(header))

#     for name, baseline_mod, efficient_mod, time_m, mem_m, flops_m in components:
#         for label, mod, CalcClass in [
#             ("baseline",  baseline_mod,  BaselineCalculator),
#             ("efficient", efficient_mod, EfficientCalculator),
#         ]:
#             mod = mod.to(device=DEVICE, dtype=DTYPE)
#             x = torch.randn(tc.batch_size, tc.seq_len, config.hidden_dim,
#                              device=DEVICE, dtype=DTYPE, requires_grad=True)
#             fn = lambda: mod(x)
#             params = list(mod.parameters())

#             actual_time = bench_time_fn(fn)
#             actual_mem = measure_mem_fn(
#                 lambda: fn().sum().backward(), exclude_tensors=params) / 1e6
            
#             with FlopCounterMode(display=False) as fc:
#                 mod(x)
#                 torch_flops = fc.get_total_flops() / 1e9

#             calc = CalcClass(mc, tc, RTX_3090)
#             pred_time = getattr(calc, time_m)()
#             pred_mem = getattr(calc, mem_m)() / 1e6
#             pred_flops = getattr(calc, flops_m)()[0]
#             tp = tflops(pred_flops, actual_time)

#             print(f"{name:<12} {label:<12} {actual_time:>10.2f} {pred_time:>10.2f} "
#                   f"{actual_mem:>10.1f} {pred_mem:>10.1f} {torch_flops:>10.2f} {pred_flops/1e9:>10.2f} {tp:>10.2f}")
#         print("-" * len(header))


# def run_all(bench_time_fn, measure_mem_fn,
#             hidden_dim=512, num_heads=8, seq_len=512,
#             layer_batch=512):
#     layer_config, layer_mc, layer_tc = make_configs(hidden_dim, num_heads, layer_batch, seq_len)
#     bench_layers(bench_time_fn, measure_mem_fn, layer_config, layer_mc, layer_tc)


# if __name__ == "__main__":
#     DEVICE = torch.device("cuda")
#     DTYPE = torch.bfloat16


#     RTX_3090 = GPUSpec(
#         name="RTX 3090",
#         memory_bandwidth_gbps=936,
#         flops_bf16=50,
#         interconnect_bandwidth_gbps=32,
#     )
#     run_all(bench_time, measure_saved_tensors_bytes)

Overwriting layer_bench.py


In [ ]:
!python benchmarks/bench_optimizer.py
!python benchmarks/bench_optimizer.py --check-kernel

In [13]:
!python benchmarks/bench_layers.py # fused cle computes everything for bwd in fwd


  Per-Layer  B=1024  S=512  H=512
Component    Type            Time ms    Pred ms     Mem MB    Pred MB   Torch GF    Calc GF    TFLOP/s
------------------------------------------------------------------------------------------------------
Attention    baseline         259.89      99.60     6980.1     6979.3    4947.80    1660.00      19.04
Attention    efficient         84.40      98.96     2701.2     2684.4    5222.68    1649.27      61.88
------------------------------------------------------------------------------------------------------
RMSNorm      baseline          47.87      30.97     2149.6     2147.5       0.00       1.07       0.00
RMSNorm      efficient          3.22       3.44      539.0      536.9       0.00       1.07       0.00
------------------------------------------------------------------------------------------------------
SwiGLU       baseline         131.74      98.96     8053.1     8053.1    4947.80    1649.27      37.56
SwiGLU       efficient         86.29  

In [14]:
!python benchmarks/bench_sweep.py 

  Single GPU
G=1 H=512 B=8 S=512
pass  type       mode        ms    pred   save   pred   peak   pred      GF    pred   TF/s
------------------------------------------------------------------------------------------
fwd   baseline   single    12.7     6.1   1192   1177    ---    ---     222     223   17.5
fb    baseline   single    39.8    18.2    ---    ---   1796   2284     665     667   16.7
step  baseline   single    44.9     ---    ---    ---    ---    ---     ---     ---    ---
G=1 H=512 B=8 S=512
pass  type       mode        ms    pred   save   pred   peak   pred      GF    pred   TF/s
------------------------------------------------------------------------------------------
fwd   efficient  single    15.1     4.6    328    310    ---    ---     356     222   23.5
fb    efficient  single    42.9    13.7    ---    ---    463    689     678     666   15.8
step  efficient  single    45.9     ---    ---    ---    ---    ---     ---     ---    ---

G=1 H=512 B=32 S=512
pass  type     

In [ ]:
!python benchmarks/bench_sweep.py --gpus 2

  Distributed (2 GPUs)
G=2 H=512 B=8 S=512
pass  type       mode        ms    pred   save   pred   peak   pred      GF    pred   TF/s
------------------------------------------------------------------------------------------
step  baseline   ddp       46.8    18.4    ---    ---   1861   2284     ---     ---    ---
  comm_pred=2.01 ms (theoretical lower bound)
G=2 H=512 B=8 S=512
pass  type       mode        ms    pred   save   pred   peak   pred      GF    pred   TF/s
------------------------------------------------------------------------------------------
step  efficient  fsdp      72.6    11.6    ---    ---    469    566     ---     ---    ---
  comm_pred=3.01 ms (theoretical lower bound)

G=2 H=512 B=32 S=512
pass  type       mode        ms    pred   save   pred   peak   pred      GF    pred   TF/s
------------------------------------------------------------------------------------------
step  baseline   ddp      132.7    67.1    ---    ---   6999   8173     ---     ---    ---
  co

In [ ]:
!python benchmarks/bench_optimizer.py --check-kernel

In [60]:
# Super cool thing to understand - fused cross entnropy calculates everything in fwd!

# h = torch.randn(16,511, 512, device='cuda', dtype=torch.bfloat16, requires_grad=True)
# w = torch.randn(16000, 512, device='cuda', dtype=torch.bfloat16, requires_grad=True)
# lab = torch.randint(0, 16000, (16,511,), device='cuda')

# from efficient_model.loss import CrossEntropyLoss
# loss_fn = CrossEntropyLoss()

# with FlopCounterMode(display=False) as fc:
#     #with torch.no_grad(): # Явно отключаем создание графа
#     #    loss = loss_fn(h, w, lab)
#     loss = loss_fn(h, w, lab)
#     #loss.sum().backward()

# print(f"with grad: {fc.get_total_flops()/1e9:.6f} GF")

In [70]:
# %%writefile bench_single.py
# """
# Single-config benchmark. Clean process, no notebook garbage.

# Usage:
#     # Single GPU
#     python bench_single.py --hidden 512 --heads 8 --batch 16 --seq 512 --type baseline
#     python bench_single.py --hidden 512 --heads 8 --batch 16 --seq 512 --type efficient

#     # Distributed (DDP wraps baseline, FSDP wraps efficient)
#     torchrun --nproc_per_node=2 bench_single.py --hidden 512 --heads 8 --batch 16 --seq 512 --type baseline
#     torchrun --nproc_per_node=2 bench_single.py --hidden 512 --heads 8 --batch 16 --seq 512 --type efficient
# """
# import os
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# import gc
# import sys
# import time
# import argparse

# import torch
# import torch.distributed as dist
# from torch.autograd.graph import saved_tensors_hooks
# from torch.utils.flop_counter import FlopCounterMode

# from config import TransformerConfig
# from model.transformer import BaselineTransformer
# from efficient_model.transformer import EfficientTransformer, TransformerBlock
# from efficient_optimizer.ademamix import AdEMAMix
# from calculators.base import ModelConfig, TrainingConfig, GPUSpec
# from calculators.baseline_calculator import BaselineCalculator
# from calculators.efficient_calculator import EfficientCalculator


# def bench_time(fn, warmup=5, iters=20):
#     for _ in range(warmup):
#         fn()
#         torch.cuda.synchronize()
#     torch.cuda.synchronize()
#     t0 = time.perf_counter()
#     for _ in range(iters):
#         fn()
#     torch.cuda.synchronize()
#     return (time.perf_counter() - t0) * 1000 / iters


# def measure_saved_tensors(fn, exclude_tensors=None):
#     exclude_ptrs = {t.data.data_ptr() for t in (exclude_tensors or [])}
#     seen = set()
#     total = 0
#     def pack(t):
#         nonlocal total
#         ptr = t.data.data_ptr()
#         if ptr not in exclude_ptrs and ptr not in seen:
#             seen.add(ptr)
#             total += t.numel() * t.element_size()
#         return t
#     with saved_tensors_hooks(pack, lambda t: t):
#         fn()
#     return total


# def measure_peak_memory(fn):
#     gc.collect()
#     torch.cuda.synchronize()
#     torch.cuda.empty_cache()
#     torch.cuda.reset_peak_memory_stats()
#     fn()
#     torch.cuda.synchronize()
#     return torch.cuda.max_memory_allocated()


# def setup_distributed():
#     if 'RANK' in os.environ:
#         dist.init_process_group(backend='nccl')
#         rank = dist.get_rank()
#         world_size = dist.get_world_size()
#         local_rank = int(os.environ.get('LOCAL_RANK', 0))
#         torch.cuda.set_device(local_rank)
#         return rank, world_size, local_rank
#     return 0, 1, 0


# def make_configs(args):
#     config = TransformerConfig(
#         hidden_dim=args.hidden, num_heads=args.heads,
#         max_seq_len=args.seq * 2, dropout=0.0,
#     )
#     mc = ModelConfig(
#         vocab_size=config.vocab_size, hidden_dim=config.hidden_dim,
#         num_heads=config.num_heads, num_layers=config.num_layers,
#         intermediate_dim=config.intermediate_dim, max_seq_len=config.max_seq_len,
#     )
#     return config, mc


# def wrap_distributed(model, world_size, local_rank, model_type):
#     if world_size == 1:
#         return model

#     if model_type == "baseline":
#         from torch.nn.parallel import DistributedDataParallel as DDP
#         return DDP(model, device_ids=[local_rank])
#     else:
#         from torch.distributed.fsdp import fully_shard
#         for layer in model.layers:
#             fully_shard(layer)
#         # fully_shard(model.lm_head) conflicts with fused cross-entropy
#         fully_shard(model)
#         return model

    
# def main():
#     parser = argparse.ArgumentParser()
#     parser.add_argument('--hidden', type=int, default=512)
#     parser.add_argument('--heads', type=int, default=8)
#     parser.add_argument('--batch', type=int, default=16)
#     parser.add_argument('--seq', type=int, default=512)
#     parser.add_argument('--type', choices=['baseline', 'efficient'], required=True)
#     args = parser.parse_args()

#     rank, world_size, local_rank = setup_distributed()
#     device = torch.device(f'cuda:{local_rank}')
#     is_master = rank == 0
#     is_distributed = world_size > 1

#     config, mc = make_configs(args)
#     tc = TrainingConfig(batch_size=args.batch, seq_len=args.seq, num_gpus=world_size)

#     ModelClass = BaselineTransformer if args.type == "baseline" else EfficientTransformer
#     CalcClass = BaselineCalculator if args.type == "baseline" else EfficientCalculator

#     model = ModelClass(config).to(device=device, dtype=DTYPE)
#     model = wrap_distributed(model, world_size, local_rank, args.type)
#     raw_model = model.module if hasattr(model, 'module') else model
#     optimizer = AdEMAMix(model.parameters(), lr=1e-4, betas=(0.9, 0.999, 0.9999))

#     input_ids = torch.randint(0, config.vocab_size, (args.batch, args.seq), device=device)
#     labels = input_ids.clone()

#     if args.type == "baseline":
#         def fwd():
#             logits = raw_model(input_ids)
#             return raw_model.compute_loss(logits, labels)
#         def fwd_bwd():
#             logits = raw_model(input_ids)
#             loss = raw_model.compute_loss(logits, labels)
#             loss.backward()
#         def full_step():
#             optimizer.zero_grad(set_to_none=True)
#             logits = raw_model(input_ids)
#             loss = raw_model.compute_loss(logits, labels)
#             loss.backward()
#             optimizer.step()
#     else:
#         def fwd():
#             return raw_model(input_ids, labels=labels)
#         def fwd_bwd():
#             loss = raw_model(input_ids, labels=labels)
#             loss.backward()
#         def full_step():
#             optimizer.zero_grad(set_to_none=True)
#             loss = raw_model(input_ids, labels=labels)
#             loss.backward()
#             optimizer.step()

#     calc = CalcClass(mc, tc, RTX_3090)

#     # FlopCounterMode adds some extra, so we need to measure peak memory first!
#     peak = measure_peak_memory(full_step)
#     torch.cuda.synchronize()
#     pred_peak = calc.calculate_peak_memory()

#     if not is_distributed:
#         params = list(model.parameters())
#         buffers = list(model.buffers())
#         exclude = params + buffers

#         saved = measure_saved_tensors(
#             lambda: fwd_bwd(), exclude_tensors=exclude)
#         pred_saved = calc.calculate_activation_memory()

#         with FlopCounterMode(display=False) as fc:
#             fwd()
#         torch_fwd_flops = fc.get_total_flops()

#         with FlopCounterMode(display=False) as fc:
#             fwd_bwd()
#         torch_fwd_bwd_flops = fc.get_total_flops()

#         pred_fwd_flops = 0
#         for _ in range(mc.num_layers):
#             pred_fwd_flops += calc._attention_breakdown()[0]
#             pred_fwd_flops += calc._mlp_breakdown()[0]
#             pred_fwd_flops += 2 * calc._rms_norm_breakdown()[0]
#         pred_fwd_flops += calc._embedding_breakdown()[0]
#         pred_fwd_flops += calc._lm_head_breakdown()[0]
#         pred_fwd_flops += calc._loss_breakdown()[0]

#         loss_flops = calc._loss_breakdown()[0]
#         pred_fwd_bwd_flops = (pred_fwd_flops - loss_flops) * 3 + loss_flops

#     if is_distributed:
#         dist.barrier()
    
#     fwd_time = bench_time(fwd)
#     fwd_bwd_time = bench_time(fwd_bwd)
#     step_time = bench_time(full_step)

#     if is_master:
#         mode = f"{'ddp' if args.type == 'baseline' else 'fsdp'}" if is_distributed else "single"

#         header = (f"{'pass':<5} {'type':<10} {'mode':<6} "
#                   f"{'ms':>7} {'pred':>7} "
#                   f"{'save':>6} {'pred':>6} "
#                   f"{'peak':>6} {'pred':>6} "
#                   f"{'GF':>7} {'pred':>7} "
#                   f"{'TF/s':>6}")
#         print(f"\nG={world_size} H={args.hidden} B={args.batch} S={args.seq}")
#         print(header)
#         print("-" * len(header))

#         if not is_distributed:
#             fwd_tfs = torch_fwd_flops / (fwd_time / 1000) / 1e12
#             bwd_tfs = torch_fwd_bwd_flops / (fwd_bwd_time / 1000) / 1e12

#             print(f"{'fwd':<5} {args.type:<10} {mode:<6} "
#                   f"{fwd_time:>7.1f} {calc.time_forward_pass_ms():>7.1f} "
#                   f"{saved/1e6:>6.0f} {pred_saved/1e6:>6.0f} "
#                   f"{'---':>6} {'---':>6} "
#                   f"{torch_fwd_flops/1e9:>7.0f} {pred_fwd_flops/1e9:>7.0f} "
#                   f"{fwd_tfs:>6.1f}")
#             print(f"{'fb':<5} {args.type:<10} {mode:<6} "
#                   f"{fwd_bwd_time:>7.1f} {calc.time_forward_backward_ms():>7.1f} "
#                   f"{'---':>6} {'---':>6} "
#                   f"{peak/1e6:>6.0f} {pred_peak/1e6:>6.0f} "
#                   f"{torch_fwd_bwd_flops/1e9:>7.0f} {pred_fwd_bwd_flops/1e9:>7.0f} "
#                   f"{bwd_tfs:>6.1f}")
#             print(f"{'step':<5} {args.type:<10} {mode:<6} "
#                   f"{step_time:>7.1f} {'---':>7} "
#                   f"{'---':>6} {'---':>6} "
#                   f"{'---':>6} {'---':>6} "
#                   f"{'---':>7} {'---':>7} "
#                   f"{'---':>6}")
#         else:
#             pred_step = calc.time_total_step_ms()
#             pred_comm = calc.time_communication_ms()
#             print(f"{'step':<5} {args.type:<10} {mode:<6} "
#                   f"{step_time:>7.1f} {pred_step:>7.1f} "
#                   f"{'---':>6} {'---':>6} "
#                   f"{peak/1e6:>6.0f} {pred_peak/1e6:>6.0f} "
#                   f"{'---':>7} {'---':>7} "
#                   f"{'---':>6}")
#             print(f"  comm_pred={pred_comm:.2f} ms")

#     if is_distributed:
#         dist.destroy_process_group()


# RTX_3090 = GPUSpec(
#     name="RTX 3090",
#     memory_bandwidth_gbps=936,
#     flops_bf16=50,
#     interconnect_bandwidth_gbps=32,
# )

# DTYPE = torch.bfloat16

# if __name__ == '__main__':
#     main()

Overwriting bench_single.py


In [76]:
# %%writefile bench_runner.py
# """
# Runs bench_single.py across configs via subprocess.

# Usage:
#     python bench_runner.py                    # single GPU only
#     python bench_runner.py --gpus 2           # single + distributed
# """
# import subprocess
# import sys
# import argparse


# CONFIGS = [
#     # (hidden, heads, batch, seq)
#     (512,  8,  8,  512),
#     (512,  8,  32, 512),
#     (512,  8,  8, 2048),
#     (512,  8,  32, 2048),
#     (1024, 8, 8,  4096),
#     (1024, 8, 32,  4096),
# ]


# def run_single(h, nh, b, s, model_type):
#     result = subprocess.run(
#         [sys.executable, "bench_single.py",
#          "--hidden", str(h), "--heads", str(nh),
#          "--batch", str(b), "--seq", str(s),
#          "--type", model_type],
#         capture_output=True, text=True,
#     )
#     if result.returncode != 0:
#         print(f"FAILED: {model_type} H={h} B={b} S={s}")
#     else:
#         print(result.stdout.strip())


# def run_distributed(h, nh, b, s, model_type, gpus):
#     result = subprocess.run(
#         ["torchrun", f"--nproc_per_node={gpus}",
#          "bench_single.py",
#          "--hidden", str(h), "--heads", str(nh),
#          "--batch", str(b), "--seq", str(s),
#          "--type", model_type],
#         capture_output=True, text=True,
#     )
#     if result.returncode != 0:
#         print(f"FAILED: {model_type} H={h} B={b} S={s}")
#     else:
#         print(result.stdout.strip())


# def main():
#     parser = argparse.ArgumentParser()
#     parser.add_argument('--gpus', type=int, default=0, help='Number of GPUs for distributed (0=skip)')
#     args = parser.parse_args()

#     if args.gpus <= 1:
#         print("=" * 80)
#         print("  Single GPU")
#         print("=" * 80)
#         for h, nh, b, s in CONFIGS:
#             for model_type in ["baseline", "efficient"]:
#                 run_single(h, nh, b, s, model_type)
#             print()
#     else:
#         print("=" * 80)
#         print(f"  Distributed ({args.gpus} GPUs)")
#         print("=" * 80)
#         for h, nh, b, s in CONFIGS:
#             # DDP = baseline, FSDP = efficient
#             run_distributed(h, nh, b, s, "baseline", args.gpus)
#             run_distributed(h, nh, b, s, "efficient", args.gpus)
#             print()


# if __name__ == '__main__':
#     main()

Overwriting bench_runner.py


In [5]:
!python benchmarks/bench_model.py --gpus=0

usage: bench_model.py [-h] [--hidden HIDDEN] [--heads HEADS] [--batch BATCH] [--seq SEQ]
bench_model.py: error: unrecognized arguments: --gpus=0


In [78]:
!python bench_runner.py --gpus=2

  Distributed (2 GPUs)
G=2 H=512 B=8 S=512
pass  type       mode        ms    pred   save   pred   peak   pred      GF    pred   TF/s
------------------------------------------------------------------------------------------
step  baseline   ddp       37.0    20.0    ---    ---   2122   2284     ---     ---    ---
  comm_pred=2.01 ms
G=2 H=512 B=8 S=512
pass  type       mode        ms    pred   save   pred   peak   pred      GF    pred   TF/s
------------------------------------------------------------------------------------------
step  efficient  fsdp      64.6    13.7    ---    ---    469    566     ---     ---    ---
  comm_pred=3.01 ms

G=2 H=512 B=32 S=512
pass  type       mode        ms    pred   save   pred   peak   pred      GF    pred   TF/s
------------------------------------------------------------------------------------------
step  baseline   ddp      131.5    73.3    ---    ---   8045   8173     ---     ---    ---
  comm_pred=2.01 ms
G=2 H=512 B=32 S=512
pass  type     

# Bench debug

In [65]:
%%writefile bench_peak_debug.py
"""
Usage: python bench_peak_debug2.py --hidden 512 --heads 8 --batch 16 --seq 2048 --type baseline
"""
import argparse
import torch
from torch.autograd.graph import saved_tensors_hooks
from config import TransformerConfig
from model.transformer import BaselineTransformer
from efficient_model.transformer import EfficientTransformer
from efficient_optimizer.ademamix import AdEMAMix
from calculators.base import ModelConfig, TrainingConfig, GPUSpec
from calculators.baseline_calculator import BaselineCalculator
from calculators.efficient_calculator import EfficientCalculator

DTYPE = torch.bfloat16
DEVICE = torch.device("cuda")
RTX_3090 = GPUSpec(name="RTX 3090", memory_bandwidth_gbps=936, flops_bf16=50, interconnect_bandwidth_gbps=32)

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--hidden', type=int, default=512)
    parser.add_argument('--heads', type=int, default=8)
    parser.add_argument('--batch', type=int, default=16)
    parser.add_argument('--seq', type=int, default=2048)
    parser.add_argument('--type', choices=['baseline', 'efficient'], default='baseline')
    args = parser.parse_args()

    config = TransformerConfig(hidden_dim=args.hidden, num_heads=args.heads, max_seq_len=args.seq * 2, dropout=0.0)
    mc = ModelConfig(
        vocab_size=config.vocab_size, hidden_dim=config.hidden_dim,
        num_heads=config.num_heads, num_layers=config.num_layers,
        intermediate_dim=config.intermediate_dim, max_seq_len=config.max_seq_len,
    )
    tc = TrainingConfig(batch_size=args.batch, seq_len=args.seq, num_gpus=1)

    ModelClass = BaselineTransformer if args.type == "baseline" else EfficientTransformer
    CalcClass = BaselineCalculator if args.type == "baseline" else EfficientCalculator

    model = ModelClass(config).to(device=DEVICE, dtype=DTYPE)
    optimizer = AdEMAMix(model.parameters(), lr=1e-4, betas=(0.9, 0.999, 0.9999))
    calc = CalcClass(mc, tc, RTX_3090)

    input_ids = torch.randint(0, config.vocab_size, (args.batch, args.seq), device=DEVICE)
    labels = input_ids.clone()

    if args.type == "baseline":
        logits = model(input_ids)
        loss = model.compute_loss(logits, labels)
    else:
        loss = model(input_ids, labels=labels)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    del loss
    if args.type == "baseline":
        del logits
    import gc; gc.collect()
    torch.cuda.empty_cache()

    # 1. Measure saved tensors
    params_ptrs = {p.data.data_ptr() for p in model.parameters()}
    buf_ptrs = {b.data.data_ptr() for b in model.buffers()}
    exclude = params_ptrs | buf_ptrs
    seen = set()
    total_saved = 0
    def pack(t):
        nonlocal total_saved
        ptr = t.data.data_ptr()
        if ptr not in exclude and ptr not in seen:
            seen.add(ptr)
            total_saved += t.numel() * t.element_size()
        return t

    optimizer.zero_grad(set_to_none=True)
    with saved_tensors_hooks(pack, lambda t: t):
        if args.type == "baseline":
            logits = model(input_ids)
            loss = model.compute_loss(logits, labels)
        else:
            loss = model(input_ids, labels=labels)
        loss.backward()
    del loss
    if args.type == "baseline":
        del logits
    gc.collect()
    torch.cuda.empty_cache()

    # 2. Measure peak
    optimizer.zero_grad(set_to_none=True)
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    static = torch.cuda.memory_allocated()

    if args.type == "baseline":
        logits = model(input_ids)
        after_fwd = torch.cuda.memory_allocated()
        peak_fwd = torch.cuda.max_memory_allocated()
        loss = model.compute_loss(logits, labels)
        after_loss = torch.cuda.memory_allocated()
        peak_loss = torch.cuda.max_memory_allocated()
    else:
        loss = model(input_ids, labels=labels)
        after_fwd = torch.cuda.memory_allocated()
        peak_fwd = torch.cuda.max_memory_allocated()
        after_loss = after_fwd
        peak_loss = peak_fwd

    loss.backward()
    optimizer.step()
    torch.cuda.synchronize()
    after_bwd = torch.cuda.memory_allocated()
    peak_bwd = torch.cuda.max_memory_allocated()

    # Print
    print(f"\n{args.type} H={args.hidden} B={args.batch} S={args.seq} V={config.vocab_size} I={config.intermediate_dim} N={config.num_layers}")
    print(f"\nActual:")
    print(f"  static:        {static/1e6:>8.0f} MB")
    if args.type == "baseline":
        print(f"  after_fwd:     {after_fwd/1e6:>8.0f} MB  peak={peak_fwd/1e6:.0f}  fwd_alloc={( after_fwd-static)/1e6:.0f}")
        print(f"  after_loss:    {after_loss/1e6:>8.0f} MB  peak={peak_loss/1e6:.0f}  loss_alloc={(after_loss-after_fwd)/1e6:.0f}")
    else:
        print(f"  after_fwd+loss:{after_fwd/1e6:>8.0f} MB  peak={peak_fwd/1e6:.0f}  fwd_alloc={(after_fwd-static)/1e6:.0f}")
    print(f"  after_bwd:     {after_bwd/1e6:>8.0f} MB  peak={peak_bwd/1e6:.0f}  bwd_transient={(peak_bwd-after_loss)/1e6:.0f}")
    print(f"  saved_tensors: {total_saved/1e6:>8.0f} MB")
    print(f"  actual_peak:   {peak_bwd/1e6:>8.0f} MB")

    print(f"\nPredicted:")
    p = calc.calculate_param_memory()
    g = calc.calculate_gradient_memory()
    o = calc.calculate_optimizer_memory()
    a = calc.calculate_activation_memory()
    t = calc._peak_transient_bytes()

    per_layer = (calc.mlp_saved_tensors_bytes() + calc.attention_saved_tensors_bytes() + 2 * calc.norm_saved_tensors_bytes())
    non_layer = calc.non_layer_saved_tensors_bytes()

    print(f"  params:        {p/1e6:>8.0f} MB")
    print(f"  grads:         {g/1e6:>8.0f} MB")
    print(f"  optim:         {o/1e6:>8.0f} MB")
    print(f"  act total:     {a/1e6:>8.0f} MB")
    print(f"    per_layer:   {per_layer/1e6:>8.0f} MB  x{config.num_layers}= {per_layer*config.num_layers/1e6:.0f}")
    print(f"      attn:      {calc.attention_saved_tensors_bytes()/1e6:>8.0f} MB")
    print(f"      mlp:       {calc.mlp_saved_tensors_bytes()/1e6:>8.0f} MB")
    print(f"      norm x2:   {2*calc.norm_saved_tensors_bytes()/1e6:>8.0f} MB")
    print(f"    non_layer:   {non_layer/1e6:>8.0f} MB")
    print(f"  transient:     {t/1e6:>8.0f} MB")
    print(f"  pred_peak:     {calc.calculate_peak_memory()/1e6:>8.0f} MB")
    print(f"\n  gap:           {(peak_bwd - calc.calculate_peak_memory())/1e6:>8.0f} MB")

    bsh = args.batch * args.seq * args.hidden * 2
    print(f"\n  B*S*H*bf16 =   {bsh/1e6:>8.0f} MB")
    print(f"  B*S*V*fp32 =   {args.batch*args.seq*config.vocab_size*4/1e6:>8.0f} MB")
    print(f"  gap / B*S*H =  {(peak_bwd - calc.calculate_peak_memory()) / bsh:>8.1f} tensors")

if __name__ == '__main__':
    main()

Overwriting bench_peak_debug.py


In [66]:
!python bench_peak_debug.py --hidden 1024 --heads 8 --batch 32 --seq 4096 --type efficient


efficient H=1024 B=32 S=4096 V=16000 I=1024 N=6

Actual:
  static:             648 MB
  after_fwd+loss:   17356 MB  peak=18418  fwd_alloc=16708
  after_bwd:          802 MB  peak=18418  bwd_transient=1063
  saved_tensors:    16709 MB
  actual_peak:      18418 MB

Predicted:
  params:             154 MB
  grads:              154 MB
  optim:              461 MB
  act total:        16645 MB
    per_layer:       2684 MB  x6= 16106
      attn:          1342 MB
      mlp:            805 MB
      norm x2:        537 MB
    non_layer:        539 MB
  transient:         1139 MB
  pred_peak:        18553 MB

  gap:               -134 MB

  B*S*H*bf16 =        268 MB
  B*S*V*fp32 =       8389 MB
  gap / B*S*H =      -0.5 tensors
